# Singing mice vocal modes - PAG TeLC experiment

## Import Libraries

In [5]:
# import libraries
import glob
import numpy as np
import os
import pandas as pd
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns

Matplotlib is building the font cache; this may take a moment.


In [6]:
# import internal libraries

## Helper Functions

In [10]:
#utils code

"""
Utility functions


"""

import numpy as np
from scipy.stats import (
    gaussian_kde,
)
from scipy.signal import (
    find_peaks,
    peak_prominences,
)
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, Circle
from scipy.interpolate import interp1d

def compute_1d_kde(data, xmin=None, xmax=None, bw_method='scott', gridsize=1000):
    """
    Compute the 1D KDE for given data and return grid values and density estimates.

    Parameters:
    - data: 1D array-like, shape (N,)
    - xmin: float or None, minimum value for the grid. If None, set to data.min() - 1.
    - xmax: float or None, maximum value for the grid. If None, set to data.max() + 1.
    - bw_method: str or scalar, bandwidth method for gaussian_kde ('scott', 'silverman', or scalar)
    - gridsize: int, number of points in the grid

    Returns:
    - x_grid: 1D array of grid points
    - density: 1D array of density estimates corresponding to grid points
    """
    # Remove NaN and Inf values
    data = np.asarray(data)
    data = data[np.isfinite(data)]
    
    if data.size == 0:
        raise ValueError("Input data must contain at least one finite value.")
    
    # Define grid boundaries
    if xmin is None:
        xmin = data.min() - 1  # Extend range for better visualization
    if xmax is None:
        xmax = data.max() + 1
    
    if xmin >= xmax:
        raise ValueError("xmin must be less than xmax.")
    
    # Perform KDE
    kde = gaussian_kde(data, bw_method=bw_method)
    
    # Define grid over specified range
    x_grid = np.linspace(xmin, xmax, gridsize)
    
    # Evaluate KDE on grid
    density = kde(x_grid)
    
    return x_grid, density


def compute_2d_kde(data, x_min, x_max, y_min, y_max, bw_method='scott', gridsize=100):
    """
    Compute the 2D KDE for given data and return grid coordinates and density values.

    Parameters:
    - data: 2D array-like, shape (2, N)
    - x_min, x_max, y_min, y_max: float, boundaries for the grid
    - bw_method: bw_method argument for gaussian_kde
    - gridsize: int, number of points in each dimension

    Returns:
    - X, Y: 2D meshgrid arrays
    - Z: 2D array of density values
    """
    # Convert data to a NumPy array
    data = np.asarray(data)

    # Validate data shape
    if data.ndim != 2 or data.shape[0] != 2:
        raise ValueError("Input 'data' must be a 2D array with shape (2, N) where N is the number of samples.")

    # Remove columns (samples) with any Inf or NaN values
    valid_mask = np.isfinite(data).any(axis=0)
    data_clean = data[:, valid_mask]

    # Check if any valid data points remain
    if data_clean.shape[1] == 0:
        raise ValueError("No valid data points remain after removing NaNs.")
    
    # Create grid
    x, y = np.mgrid[x_min:x_max:gridsize*1j, y_min:y_max:gridsize*1j]
    positions = np.vstack([x.ravel(), y.ravel()])
    
    # Perform KDE
    kernel = gaussian_kde(data_clean, bw_method=bw_method)
    
    Z = np.reshape(kernel(positions).T, x.shape)
    
    return x, y, Z

def find_threshold_between_peaks(x, density, num_peaks=2):
    # Find all peaks in the density
    peaks, _ = find_peaks(density)
    
    if len(peaks) < num_peaks:
        raise ValueError(f"Expected at least {num_peaks} peaks, but found {len(peaks)}.")
    
    # Compute prominences of all peaks
    prominences = peak_prominences(density, peaks)[0]
    
    # Sort peaks based on prominence
    sorted_indices = np.argsort(prominences)[::-1]
    top_peaks = peaks[sorted_indices[:num_peaks]]
    top_peaks = np.sort(top_peaks)
    
    if len(top_peaks) < num_peaks:
        raise ValueError(f"Could not identify {num_peaks} distinct peaks in the distribution.")
    
    # Find the valley between the two top peaks
    peak1, peak2 = top_peaks[:num_peaks]
    if peak1 == peak2:
        raise ValueError("The two top peaks are the same. Unable to determine a valley.")
    
    # Ensure peak1 is to the left of peak2
    if peak1 > peak2:
        peak1, peak2 = peak2, peak1
    
    # Extract the region between the two peaks
    density_between = density[peak1:peak2]
    x_between = x[peak1:peak2]
    
    # Find the minimum in this region
    valley_index = np.argmin(density_between) + peak1
    threshold = x[valley_index]
    
    return threshold, x[valley_index], valley_index

def draw_ellipse(position, covariance, ax, n_std=1.0, **kwargs):
    """
    Draw an ellipse representing the covariance matrix centered at position.
    `position`: The center of the ellipse (mean of the Gaussian).
    `covariance`: 2x2 covariance matrix of the Gaussian.
    `ax`: Matplotlib Axes object to draw the ellipse on.
    `n_std`: The number of standard deviations for the ellipse radius.
    Additional keyword arguments are passed to Ellipse patch.
    """
    if covariance.shape == (2, 2):
        eigenvalues, eigenvectors = np.linalg.eigh(covariance)
        order = eigenvalues.argsort()[::-1]
        eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]
        angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
        
        # Adjust width and height for the desired number of std deviations
        width, height = 2 * n_std * np.sqrt(eigenvalues)
    else:
        angle = 0
        width = height = 2 * n_std * np.sqrt(covariance)

    ell = Ellipse(xy=position, width=width, height=height, angle=angle, **kwargs)
    ax.add_patch(ell)

def plot_skeleton(ax, location_frame, skeleton, color):

    xs = location_frame[:,0].astype(float)
    ys = location_frame[:,1].astype(float)

    # plot joints
    for i_joint in range(skeleton.shape[0]):
        index1, index2 = skeleton[i_joint, 0], skeleton[i_joint, 1]
        ax.plot([xs[index1], xs[index2]], 
                [ys[index1], ys[index2]],
                color=color, linewidth=1, linestyle='-')

    # plot points
    for j in range(location_frame.shape[0]):
        ax.add_patch(Circle([xs[j], ys[j]], radius=2, color=color, zorder=2))


def fill_missing(Y, kind="linear"):
    """Fills missing values independently along each dimension after the first."""

    # Store initial shape.
    initial_shape = Y.shape

    # Flatten after first dim.
    Y = Y.reshape((initial_shape[0], -1))

    # Interpolate along each slice.
    for i in range(Y.shape[-1]):
        y = Y[:, i]

        # Build interpolant.
        x = np.flatnonzero(~np.isnan(y))
        f = interp1d(x, y[x], kind=kind, fill_value=np.nan, bounds_error=False)

        # Fill missing
        xq = np.flatnonzero(np.isnan(y))
        y[xq] = f(xq)
        
        # Fill leading or trailing NaNs with the nearest non-NaN values
        mask = np.isnan(y)
        y[mask] = np.interp(np.flatnonzero(mask), np.flatnonzero(~mask), y[~mask])

        # Save slice
        Y[:, i] = y

    # Restore to initial shape.
    Y = Y.reshape(initial_shape)

    return Y

def compute_moving_event_rate(events_df, start_time=0, end_time=3600, window_size=10, step_size=10):
    """
    Computes the event rate (counts) in a sliding window over the duration of a session,
    returning NumPy arrays for window start times and counts.
    
    Parameters:
        events_df (pd.DataFrame): DataFrame containing event data with a 'start' column.
        start_time (float): Start time of the session.
        end_time (float): End time of the session.
        window_size (float): Size of the sliding window in seconds.
        step_size (float): Step size to slide the window in seconds.
        
    Returns:
        window_starts (np.ndarray): Array of window start times.
        counts (np.ndarray): Array of event counts in each window.
    """
    # Create an array of window start times from start_time to end_time such that window fits within end_time
    window_starts = np.arange(start_time, end_time - window_size + step_size, step_size)
    window_midpoints = window_starts + window_size / 2.0
    counts = np.empty_like(window_starts, dtype=int)
    
    # Loop through each window and count events
    for i, t in enumerate(window_starts):
        window_end = t + window_size
        # Count events where the start time falls within the current window
        count = events_df[(events_df['start'] >= t) & (events_df['start'] < window_end)].shape[0]
        counts[i] = count
        
    return window_midpoints, counts

In [13]:
#events.py copied from their github

"""
    Temporal patterning analysis of events

"""

import numpy as np
%pip install scikit-learn
import pandas as pd
from sklearn.linear_model import LinearRegression, RANSACRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from scipy.stats import linregress  


def compute_event_features(df):
    """
    Compute per-event temporal features for behavioral events in a DataFrame.

    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing at least 'start' and 'end' columns representing event timings.

    Returns:
    --------
    pandas.DataFrame
        DataFrame augmented with the following per-event temporal features:
            - duration: Duration of each event (end - start)
            - inter_event_interval: Time between the end of the current event and the start of the next event
            - log_inter_event_interval: Log10 of the inter-event interval
            - inter_start_interval: Time between the start of the current event and the start of the next event
            - inst_rate: Instantaneous rate (1 / inter_start_interval)
    """
    df = df.copy()
    
    # Ensure that 'start' and 'end' columns exist
    if not {'start', 'end'}.issubset(df.columns):
        raise ValueError("Input DataFrame must contain 'start' and 'end' columns.")
    
    # Sort the DataFrame by 'start' time to ensure chronological order
    df = df.sort_values(by='start').reset_index(drop=True)
    
    ### Per-Event Temporal Features ###
    
    # 1. Compute event duration
    df['duration'] = df['end'] - df['start']
    
    # Handle cases where 'end' is before 'start'
    negative_durations = df['duration'] < 0
    if negative_durations.any():
        raise ValueError("Some events have 'end' times earlier than 'start' times.")
    
    # 2. Compute inter-event interval (next_start - current_end)
    df['inter_event_interval'] = df['start'].shift(-1) - df['end']
    
    # 3. Compute log inter-event interval (log10)
    # Replace non-positive intervals with NaN to avoid log of zero or negative
    df['log_inter_event_interval'] = np.log10(df['inter_event_interval'].replace({0: np.nan}))
    
    # 4. Compute inter-start interval (next_start - current_start)
    df['inter_start_interval'] = df['start'].shift(-1) - df['start']
    
    # 5. Compute instantaneous rate (1 / inter_start_interval)
    df['inst_rate'] = 1 / df['inter_start_interval']
    
    # Handle division by zero or negative intervals in inst_rate
    df['inst_rate'] = df['inst_rate'].replace([np.inf, -np.inf], np.nan)
    
    # # Optional: Round numerical columns for readability
    # df[['duration', 'inter_event_interval', 'log_inter_event_interval', 
    #     'inter_start_interval', 'inst_rate']] = df[
    #         ['duration', 'inter_event_interval', 'log_inter_event_interval', 
    #          'inter_start_interval', 'inst_rate']
    #     ].round(6)
    
    return df

def compute_window_features(df, min_window_size=7):
    """
    Compute windowed temporal features for behavioral events in a DataFrame.

    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing per-event temporal features, including 'inst_rate' and 'log_inter_event_interval'.
    min_window_size : int, optional
        The size of the sliding window (number of events), by default 7.
        The regression will be performed on the first (min_window_size - 1) events within each window.

    Returns:
    --------
    pandas.DataFrame
        DataFrame where each row corresponds to a sliding window with the following columns:
            - start: Start time of the first event in the window
            - end: End time of the last event in the window
            - start_index: Index of the first event in the window
            - end_index: Index of the last event in the window
            - num_notes: Number of events in the window (should be min_window_size)
            - window_max_log_inter_event_interval: Maximum log inter-event interval within the first (min_window_size - 1) events
            - window_rate_index_slope: Slope of instantaneous rate over event indices within the first (min_window_size - 1) events
            - window_rate_index_intercept: Intercept of instantaneous rate over event indices within the first (min_window_size - 1) events
            - window_rate_index_normalized_rmse: Normalized Root Mean Square Error of the regression fit (RMSE / y_mean)
    """
    df = df.copy()
    
    # Automatically set regression_window_size
    regression_window_size = min_window_size - 1
    
    # Validate that regression_window_size is at least 1
    if regression_window_size < 1:
        raise ValueError("min_window_size must be at least 2 to perform regression on at least one event.")
    
    # Ensure that required columns exist
    required_columns = {'start', 'end', 'inst_rate', 'log_inter_event_interval'}
    if not required_columns.issubset(df.columns):
        missing = required_columns - set(df.columns)
        raise ValueError(f"Input DataFrame is missing required columns: {missing}")
    
    # Sort the DataFrame by 'start' time to ensure chronological order
    df = df.sort_values(by='start').reset_index(drop=True)
    
    ### Windowed Temporal Features ###
    
    # Initialize lists to store windowed features
    window_start_times = []
    window_end_times = []
    start_indices = []
    end_indices = []
    num_notes_list = []
    window_max_log_inter_event_interval = []
    window_rate_index_slope = []
    window_rate_index_intercept = []
    window_rate_index_normalized_rmse = []
    
    # Number of windows
    num_windows = len(df) - min_window_size + 1
    
    for i in range(num_windows):
        window_df = df.iloc[i:i+min_window_size]
        
        # Define the regression subset (first min_window_size - 1 events)
        regression_df = window_df.iloc[:regression_window_size]
        
        # Check for NaN values in inst_rate within regression_df
        if regression_df['inst_rate'].isnull().any():
            # Skip this window if any inst_rate is NaN within regression window
            continue
        
        # Extract inst_rate and event indices for regression
        X = np.arange(regression_window_size).reshape(-1, 1)  # Event indices: 0,1,...,min_window_size-2
        y = regression_df['inst_rate'].values  # Actual inst_rate
        
        # Compute y_mean
        y_mean = y.mean()
        if y_mean == 0:
            # Skip this window to avoid division by zero
            continue
        
        # Fit linear regression: inst_rate ~ event index
        model = LinearRegression()
        model.fit(X, y)
        y_pred = model.predict(X)
        
        # Compute slope and intercept
        slope = model.coef_[0]
        intercept = model.intercept_
        
        # Compute RMSE
        rmse = np.sqrt(mean_squared_error(y, y_pred))
        
        # Compute Normalized RMSE
        normalized_rmse = rmse / y_mean
        
        # Compute window_max_log_inter_event_interval (from the regression subset)
        max_log_interval = regression_df['log_inter_event_interval'].max()
        
        # Append to lists
        window_start_times.append(window_df['start'].iloc[0])
        window_end_times.append(window_df['end'].iloc[-1])
        start_indices.append(i)
        end_indices.append(i + min_window_size - 1)
        num_notes_list.append(min_window_size)
        window_max_log_inter_event_interval.append(max_log_interval)
        window_rate_index_slope.append(slope)
        window_rate_index_intercept.append(intercept)
        window_rate_index_normalized_rmse.append(normalized_rmse)
    
    # Create the windowed DataFrame
    df_windowed_features = pd.DataFrame({
        'start': window_start_times,
        'end': window_end_times,
        'start_index': start_indices,
        'end_index': end_indices,
        'num_notes': num_notes_list,
        'window_max_log_inter_event_interval': window_max_log_inter_event_interval,
        'window_rate_index_slope': window_rate_index_slope,
        'window_rate_index_intercept': window_rate_index_intercept,
        'window_rate_index_normalized_rmse': window_rate_index_normalized_rmse
    })
    
    # Optional: Reset index for cleanliness
    df_windowed_features = df_windowed_features.reset_index(drop=True)
    
    return df_windowed_features

def merge_stereotypy_windows(
    df_windowed_features, 
    df_event_features,
    max_log_interval_threshold, 
    rmse_threshold, 
    max_gap,
    max_index_gap,
):
    """
    Merge fixed-size windows into larger segments based on "local stereotypy" criteria with relaxed merging conditions.
    
    A window is considered to exhibit "local stereotypy" if:
        - `window_max_log_inter_event_interval` <= `max_log_interval_threshold`
        - `window_rate_index_normalized_rmse` <= `rmse_threshold`
    
    Consecutive windows that meet these criteria are merged into a single segment if:
        - **Overlapping:** The current window overlaps with the previous window (i.e., the gap in time is negative).
        - **Non-Overlapping:** If not overlapping, then:
            - The gap between the previous window's `end` time and the current window's `start` time is <= `max_gap`
            - The gap between the previous window's `end_index` and the current window's `start_index` is <= `max_index_gap`
    
    Additionally, each merged segment records the number of events it contains and the maximum smoothed instantaneous rate within the segment.
    
    Parameters:
    -----------
    df_windowed_features : pandas.DataFrame
        DataFrame containing windowed temporal features, including:
            - `start`: Start time of the window
            - `end`: End time of the window
            - `start_index`: Index of the first event in the window
            - `end_index`: Index of the last event in the window
            - `window_max_log_inter_event_interval`: Maximum log inter-event interval within the window
            - `window_rate_index_normalized_rmse`: RMSE of the regression fit within the window
            - `window_rate_index_slope`: Slope from the regression fit within the window
            - `window_rate_index_intercept`: Intercept from the regression fit within the window
    
    df_event_features : pandas.DataFrame
        DataFrame containing per-event temporal features, including:
            - `start`: Start time of each event
            - `end`: End time of each event
            - Other features computed by `compute_event_features`
    
    max_log_interval_threshold : float
        Threshold for `window_max_log_inter_event_interval`. Windows with values <= this threshold are considered for merging.
    
    rmse_threshold : float
        Threshold for `window_rate_index_normalized_rmse`. Windows with values <= this threshold are considered for merging.
    
    max_gap : float
        Maximum allowed time gap (in time units) between consecutive windows for them to be merged if they do not overlap.
    
    max_index_gap : int
        Maximum allowed gap (in event indices) between consecutive windows for them to be merged if they do not overlap.
    
    Returns:
    --------
    tuple of pandas.DataFrame
        - `df_filtered`: The filtered DataFrame of windows that meet the merging criteria.
        - `df_segments`: DataFrame where each row corresponds to a merged segment with the following columns:
            - `start`: Start time of the first window in the segment
            - `end`: End time of the last window in the segment
            - `start_index`: Start index of the first window in the segment
            - `end_index`: End index of the last window in the segment
            - `num_events`: Number of events contained within the segment
    """
    # Create a copy of the input DataFrame to avoid modifying the original data
    df = df_windowed_features.copy()
    
    # Filter windows that meet the local stereotypy criteria based on log interval and RMSE thresholds
    df_filtered = df[
        (df['window_max_log_inter_event_interval'] <= max_log_interval_threshold) &
        (df['window_rate_index_normalized_rmse'] <= rmse_threshold)
    ].sort_values(by='start_index').reset_index(drop=True)

    # Initialize a list to store the merged segments
    segments = []
    
    # Initialize variables to keep track of the current segment and the previous window
    current_segment = None
    previous_window = None
    
    # Iterate over each window in the filtered DataFrame
    for _, window in df_filtered.iterrows():
        if current_segment is None:
            # Initialize a new segment with the current window's details
            current_segment = {
                'segment_start_time': window['start'],
                'segment_end_time': window['end'],
                'segment_start_index': window['start_index'],
                'segment_end_index': window['end_index'],
            }
        else:
            # Calculate the gap between the previous window's end time and the current window's start time
            gap_time = window['start'] - previous_window['end']
            
            # Calculate the gap in event indices between the previous window's end_index and the current window's start_index
            gap_index = window['start_index'] - previous_window['end_index'] - 1  # -1 because end_index is inclusive
            
            # Determine if the current window overlaps with the previous window
            if gap_time < 0:
                # Overlapping windows are merged unconditionally
                can_merge = True
            else:
                # Non-overlapping windows are merged only if both time and index gaps are within thresholds
                can_merge = (gap_time <= max_gap) and (gap_index <= max_index_gap)
            
            if can_merge:
                # Merge into current segment
                current_segment['segment_end_time'] = window['end']
                current_segment['segment_end_index'] = window['end_index']
            else:
                # Finalize the current segment and add it to the segments list
                segments.append({
                    'start': current_segment['segment_start_time'],
                    'end': current_segment['segment_end_time'],
                    'start_index': current_segment['segment_start_index'],
                    'end_index': current_segment['segment_end_index'],
                })
                
                # Initialize a new segment with the current window's details
                current_segment = {
                    'segment_start_time': window['start'],
                    'segment_end_time': window['end'],
                    'segment_start_index': window['start_index'],
                    'segment_end_index': window['end_index'],
                }
        
        # Update the reference to the previous window for the next iteration
        previous_window = window
    
    # After processing all windows, ensure the last segment is added to the segments list
    if current_segment is not None:
        segments.append({
            'start': current_segment['segment_start_time'],
            'end': current_segment['segment_end_time'],
            'start_index': current_segment['segment_start_index'],
            'end_index': current_segment['segment_end_index'],
        })
    
    # Convert the list of segments into a DataFrame
    if len(segments) > 0:
        df_segments = pd.DataFrame(segments)
    else:
        df_segments = pd.DataFrame(columns=['start', 'end', 'start_index', 'end_index'])
    
    # Calculate the number of events in each segment
    df_segments['num_events'] = df_segments['end_index'] - df_segments['start_index'] + 1

    # Also compute the duration of each segment
    df_segments['duration'] = df_segments['end'] - df_segments['start']
    
    return df_filtered, df_segments

def characterize_segments(df_events, df_segments, 
                          smoothing_window_size=5,
                          ransac_min_samples=9,
                          ransac_normalized_residual_threshold=0.15,
                          plot_ransac=False):
    """
    Characterize merged segments by computing the maximum smoothed instantaneous rate
    and applying RANSAC regression to identify linear patterns in song-like segments.

    Parameters:
    -----------
    df_events : pandas.DataFrame
        DataFrame containing per-event temporal features, including:
            - 'start': Start time of each event.
            - 'end': End time of each event.
            - 'inst_rate': Instantaneous rate of each event.
            - Other features as computed by compute_event_features().
    df_segments : pandas.DataFrame
        DataFrame containing merged segments with columns:
            - 'start': Start time of the segment.
            - 'end': End time of the segment.
            - 'start_index': Index of the first event in the segment.
            - 'end_index': Index of the last event in the segment.
            - 'num_events': Number of events in the segment.
            - Other features as computed by merge_stereotypy_windows().
    smoothing_window_size : int, optional
        Window size for the rolling median. Defaults to 5.
    song_like_pattern_min_max_rate : float, optional
        Minimum maximum smoothed instantaneous rate to consider a segment as song-like. Defaults to 15.
    ransac_min_samples : int, optional
        The minimum number of samples required to fit the RANSAC model. Defaults to 9.
    ransac_normalized_residual_threshold : float, optional
        Normalized residual threshold for RANSAC to classify inliers. Defaults to 0.15.

    Returns:
    --------
    pandas.DataFrame
        DataFrame where each row corresponds to a segment with additional columns:
            - 'max_smoothed_inst_rate': Maximum smoothed instantaneous rate within the segment.
            - 'is_song_like': Boolean flag indicating if the segment is song-like.
            - 'ransac_slope': Slope of the RANSAC-fitted line (NaN if not applicable).
            - 'ransac_intercept': Intercept of the RANSAC-fitted line (NaN if not applicable).
            - 'ransac_max_rate': Maximum predicted instantaneous rate from RANSAC inliers (NaN if not applicable).
            - 'ransac_normalized_rmse': Normalized RMSE of the RANSAC fit (NaN if not applicable).
            - 'ransac_inlier_count': Number of inliers identified by RANSAC (0 if not applicable).
            - 'ransac_start': Start time of the inlier segment (NaN if not applicable).
            - 'ransac_end': End time of the inlier segment (NaN if not applicable).
    """
    
    df_segments = df_segments.copy()

    ## STEP 1: compute the maximum smoothed instantaneous rate for each segment

    # Initialize the list to store maximum smoothed instantaneous rates
    max_smoothed_inst_rates = []

    # Iterate through each segment in df_segments
    for i, row in df_segments.iterrows():
        # Extract events that fall within the current segment
        events_in_segment = df_events[
            (df_events['start'] >= row['start']) & 
            (df_events['end'] <= row['end'])
        ]
        
        if events_in_segment.empty:
            # If no events fall within the segment, append NaN
            max_smoothed_inst_rates.append(np.nan)
            continue
        
        # Define Median filtering parameters
        window_size = smoothing_window_size  # e.g., 5

        # Apply Median filtering to 'inst_rate'
        # Note: 'inst_rate' for the last event in the segment might not have a following event, hence iloc[:-1]
        smoothed_inst_rate = events_in_segment['inst_rate'].iloc[:-1].rolling(
            window=window_size, 
            min_periods=1, 
            center=True
        ).median()
        
        # Determine the maximum smoothed inst_rate
        max_smoothed_inst_rate = smoothed_inst_rate.max()
        
        # Append the result to the list
        max_smoothed_inst_rates.append(max_smoothed_inst_rate)

    # Optionally, you can add the results back to your df_segments DataFrame
    df_segments['max_smoothed_inst_rate'] = max_smoothed_inst_rates

    ## STEP 2: Apply RANSAC to identify robust linear patterns

    # Initialize lists to store RANSAC results
    ransac_slopes = []
    ransac_intercepts = []
    ransac_max_rates = []
    ransac_min_rates = []
    ransac_normalized_rmses = []
    ransac_inlier_counts = []
    ransac_start_idxs = []
    ransac_end_idxs = []
    ransac_starts = []
    ransac_ends = []

    for i, row in df_segments.iterrows():
        events_in_segment = df_events[
            (df_events['start'] >= row['start']) & 
            (df_events['end'] <= row['end'])
        ].reset_index(drop=True)

        # Apply RANSAC
        # Define the regression subset (first min_window_size - 1 events)
        regression_window_size = len(events_in_segment) - 1
        regression_df = events_in_segment.iloc[:regression_window_size]
        
        # Extract inst_rate and event indices for regression
        X = np.arange(regression_window_size).reshape(-1, 1)  # Event indices: 0,1,...,min_window_size-2
        y = regression_df['inst_rate'].values  # Actual inst_rate

        # print(f'RANSAC residual threshold: {np.mean(y)*ransac_normalized_residual_threshold}')

        ransac = RANSACRegressor(
            min_samples=ransac_min_samples-1,
            residual_threshold=np.mean(y)*ransac_normalized_residual_threshold,
            random_state=42,
        )

        try:
            # Fit the model
            ransac.fit(X, y)
            # print(f'RANSAC TRIALS: {ransac.n_trials_}')
            
            # Predict using the fitted model
            y_pred = ransac.predict(X)

            # Identify inliers and outliers
            inlier_mask = ransac.inlier_mask_
            outlier_mask = ~inlier_mask

            # Calculate inlier count
            inlier_count = np.sum(inlier_mask)
            # Extract slope and intercept
            slope = ransac.estimator_.coef_[0]
            intercept = ransac.estimator_.intercept_
            ransac_max_rate = np.max(y_pred[inlier_mask])
            ransac_min_rate = np.min(y_pred[inlier_mask])
            # Compute RMSE
            rmse = np.sqrt(mean_squared_error(y[inlier_mask], y_pred[inlier_mask]))
            normalized_rmse = rmse / np.mean(y[inlier_mask])

            # get the preperties of the inlier segment
            ransac_start_idx = np.where(inlier_mask)[0][0]
            ransac_end_idx = np.where(inlier_mask)[0][-1]+1
            ransac_start = events_in_segment.iloc[X[inlier_mask].reshape(-1)[0]]['start']
            ransac_end = events_in_segment.iloc[X[inlier_mask].reshape(-1)[-1]+1]['end']

            # plot ransac diagnostic plots
            if plot_ransac:
                fig, ax = plt.subplots(figsize=(4,3))
                ax.scatter(X[inlier_mask], y[inlier_mask], color="tab:blue", marker=".", label="Inliers")
                ax.scatter(X[outlier_mask], y[outlier_mask], color="tab:orange", marker=".", label="Outliers")
                ax.plot(X[inlier_mask], y_pred[inlier_mask], color='red', label='RANSAC fit')
                ax.axvline(ransac_start_idx, color='r', linestyle='--')
                ax.axvline(ransac_end_idx, color='r', linestyle='--')
                ax.axvspan(0, regression_window_size, color='gray', linewidth=0, alpha=0.5)
                ax.set_xlabel('Note index')
                ax.set_ylim([0, 25])
                ax.set_ylabel('Note rate (Hz)')
                plt.legend(loc='upper right', fontsize='small')
                plt.tight_layout()
                plt.show()

            # Store the results
            ransac_slopes.append(slope)
            ransac_intercepts.append(intercept)
            ransac_max_rates.append(ransac_max_rate)
            ransac_min_rates.append(ransac_min_rate)
            ransac_normalized_rmses.append(normalized_rmse)
            ransac_inlier_counts.append(inlier_count)
            ransac_start_idxs.append(ransac_start_idx)
            ransac_end_idxs.append(ransac_end_idx)
            ransac_starts.append(ransac_start)
            ransac_ends.append(ransac_end)

        except Exception as e:
        # In case of any exception during fitting, return NaNs and False
            ransac_slopes.append(np.nan)
            ransac_intercepts.append(np.nan)
            ransac_max_rates.append(np.nan)
            ransac_min_rates.append(np.nan)
            ransac_normalized_rmses.append(np.nan)
            ransac_inlier_counts.append(0)
            ransac_start_idxs.append(np.nan)
            ransac_end_idxs.append(np.nan)
            ransac_starts.append(np.nan)
            ransac_ends.append(np.nan)

    # Add RANSAC results to df_segments
    df_segments['ransac_slope'] = ransac_slopes
    df_segments['ransac_intercept'] = ransac_intercepts
    df_segments['ransac_max_rate'] = ransac_max_rates
    df_segments['ransac_min_rate'] = ransac_min_rates
    df_segments['ransac_normalized_rmse'] = ransac_normalized_rmses
    df_segments['ransac_inlier_count'] = ransac_inlier_counts
    df_segments['ransac_start_index'] = ransac_start_idxs
    df_segments['ransac_end_index'] = ransac_end_idxs
    df_segments['ransac_start'] = ransac_starts
    df_segments['ransac_end'] = ransac_ends

    df_segments['ransac_duration'] = df_segments['ransac_end'] - df_segments['ransac_start']
    df_segments['ransac_num_events'] = df_segments['ransac_end_index'] - df_segments['ransac_start_index'] + 1

    return df_segments

def classify_segments(df_segments, scaler, pca, gmm, cluster_mapping):

    df_segments = df_segments.copy()  # To avoid modifying the original dataframe

    if df_segments.empty:
        # Optionally add an empty 'cluster' column or simply return the original DataFrame
        df_segments['cluster'] = []  
        return df_segments

    # Define the feature columns expected by the models
    features = ['ransac_num_events', 'ransac_max_rate', 'ransac_min_rate']
    
    # Ensure the dataframe contains the required features
    missing_features = [feat for feat in features if feat not in df_segments.columns]
    if missing_features:
        raise ValueError(f"Missing required features: {missing_features}")
    
    # Select and copy the relevant data
    data = df_segments[features].copy()
    
    # Apply the scaler and PCA transformations
    X_scaled = scaler.transform(data)
    X_pca = pca.transform(X_scaled)
    
    # Predict cluster labels using the GMM model
    labels = gmm.predict(X_pca)
    
    # Add the cluster labels as a new column in the dataframe
    df_segments['cluster'] = labels
    df_segments['cluster_label'] = df_segments['cluster'].map(cluster_mapping)
    
    return df_segments

def classify_notes_by_temporal_patterning(df_segments, df_notes):
    # Make a copy of df_notes to avoid modifying the original data
    df_notes = df_notes.copy()

    if df_notes.empty:
        df_notes['type_patterning'] = []  
        return df_notes
    
    # Initialize the new column with default value 'U' (for Unclassified)
    df_notes['type_patterning'] = 'U'
    
    # Iterate through each note
    for note_idx, note in df_notes.iterrows():
        note_start = note['start']
        note_end = note['end']
        
        # Look for a segment that completely contains the note
        for _, segment in df_segments.iterrows():
            if segment['ransac_start'] <= note_start and segment['ransac_end'] >= note_end:
                # Found a segment containing the note
                cluster_label = segment['cluster_label']
                df_notes.at[note_idx, 'type_patterning'] = cluster_label
                # Once a matching segment is found, break out of the inner loop
                break
                
    return df_notes

def classify_notes_by_amplitude(df_notes, amp_high_threshold):
    # Create a copy to avoid modifying the original dataframe
    df_notes = df_notes.copy()

    if df_notes.empty:
        df_notes['type_amplitude'] = []  
        return df_notes
    
    # Check if the required column exists
    if 'max_amp_high' not in df_notes.columns:
        raise ValueError("The dataframe must contain a 'max_amp_high' column.")
    
    # Classify each note based on the threshold
    df_notes['type_amplitude'] = df_notes['max_amp_high'].apply(
        lambda amp: "loud" if amp >= amp_high_threshold else "soft"
    )
    
    return df_notes

def find_vocal_bouts(df_notes, gap_threshold):
    """
    Merge consecutive intervals in df_notes separated by gaps <= gap_threshold.
    
    Parameters:
    - df_notes (pd.DataFrame): DataFrame containing 'start' and 'end' columns.
    - gap_threshold (float): Maximum allowed gap between consecutive intervals to merge them.
    
    Returns:
    - vocal_bouts (pd.DataFrame): DataFrame of merged intervals with additional info.
    - isolated_notes (pd.DataFrame): DataFrame of original intervals not merged into any vocal bout.
    """
    
    if df_notes.empty:
        # Return empty DataFrames if input is empty
        vocal_bouts = pd.DataFrame(columns=['start', 'end', 'sum_duration', 'num_notes'])
        isolated_notes = pd.DataFrame(columns=['start', 'end'])
        return vocal_bouts, isolated_notes
    
    # Create a copy of the input DataFrame to avoid modifying the original data
    df_notes = df_notes.copy()

    # Sort intervals by start time
    intervals_sorted = df_notes.sort_values(by='start').reset_index(drop=True)
    
    # Convert to list of lists for easier manipulation
    intervals = intervals_sorted[['start', 'end']].values.tolist()
    
    # Initialize merged intervals with the first interval
    merged_intervals = [intervals[0].copy()]
    sum_durations = [intervals[0][1] - intervals[0][0]]
    num_notes = [1]
    
    # Iterate through the sorted intervals and merge based on gap_threshold
    for current in intervals[1:]:
        last_merged = merged_intervals[-1]
        gap = current[0] - last_merged[1]
        
        if gap <= gap_threshold:
            # Merge the current interval with the last merged interval
            merged_intervals[-1][1] = current[1]
            sum_durations[-1] += current[1] - current[0]
            num_notes[-1] += 1
        else:
            # No merge needed; add the current interval as a new entry
            merged_intervals.append(current.copy())
            sum_durations.append(current[1] - current[0])
            num_notes.append(1)
    
    # Create a DataFrame for merged intervals
    merged_df = pd.DataFrame(merged_intervals, columns=['start', 'end'])
    merged_df['sum_duration'] = sum_durations
    merged_df['num_notes'] = num_notes
    merged_df['duration'] = merged_df['end'] - merged_df['start']
    
    # Separate merged intervals into vocal_bouts and isolated_notes
    vocal_bouts = merged_df[merged_df['num_notes'] > 1].reset_index(drop=True)
    isolated_merged = merged_df[merged_df['num_notes'] == 1].reset_index(drop=True)
    
    # To find isolated_notes, merge the original intervals that correspond to isolated_merged
    # Since isolated_merged corresponds to single original intervals, we can match them
    # based on start and end times
    if not isolated_merged.empty:
        # Merge based on 'start' and 'end' to ensure correct matching
        isolated_notes = pd.merge(isolated_merged[['start', 'end']], intervals_sorted, on=['start', 'end'], how='left')
    else:
        # If there are no isolated merged intervals, return an empty DataFrame
        isolated_notes = pd.DataFrame(columns=['start', 'end'])
    
    return vocal_bouts, isolated_notes

def analyze_song_patterning(df_song_segments, df_notes, song_final_note_occupancy=0.5):

    # Make a copy of the input DataFrames to avoid modifying the original data
    df_song_segments = df_song_segments.copy()
    df_notes = df_notes.copy()

    if df_song_segments.empty:
        # Return an empty DataFrame if there are no song segments
        return pd.DataFrame()
    
    songs = []
    for i_segment, song_segment in df_song_segments.iterrows():

        # Record song start and end times
        t_song_start = song_segment['ransac_start']
        t_song_end = song_segment['ransac_end']
        t_song_duration = t_song_end - t_song_start

        # Extract notes in song
        notes_in_song = df_notes[
            (df_notes['start'] >= t_song_start) & 
            (df_notes['end'] <= t_song_end)
        ]

        # Compute song patterning features
        xs = notes_in_song.index[:-1]-notes_in_song.index[0]
        ys = notes_in_song['inst_rate'].iloc[:-1].values

        slope, intercept, r_value, _, _ = linregress(xs, ys)
        y_preds = xs * slope + intercept
        rmse = np.sqrt(mean_squared_error(ys, y_preds))
        nrmse = rmse/np.mean(ys)
        rate_idx_rmax = 0 * slope + intercept
        rate_idx_rmin = (len(notes_in_song)-1) * slope + intercept

        # get the data r_min and r_max
        data_r_start, data_r_end = ys[0], ys[-1]

        # pred song duration from slope, r_min, r_max
        t_song_duration_pred = 1 / slope * np.log(rate_idx_rmin / rate_idx_rmax)
        # add an estimated last note duration
        t_song_duration_pred += 1 / rate_idx_rmin * song_final_note_occupancy

        # find longest note
        longest_note = notes_in_song.loc[notes_in_song['duration'].idxmax()]
        longest_note_t_start = longest_note['start']
        longest_note_duration = longest_note['duration']

        songs.append({
            'start': t_song_start,
            'end': t_song_end,
            'duration': t_song_duration,
            'num_notes': len(notes_in_song),
            'data_r_start': data_r_start,
            'data_r_end': data_r_end,
            'rate_idx_slope': slope,
            'rate_idx_intercept': intercept,
            'rate_idx_rmax': rate_idx_rmax,
            'rate_idx_rmin': rate_idx_rmin,
            'rate_idx_normalized_rmse': nrmse,
            'rate_idx_r2': r_value**2,
            'duration_pred': t_song_duration_pred,
            'longest_note_t_start': longest_note_t_start,
            'longest_note_duration': longest_note_duration,
        })

    df_songs = pd.DataFrame(songs)

    return df_songs

def get_song_social_context(df_songs, df_notes, context_duration=10):

    df_songs = df_songs.copy()
    
    num_notes_pre_song = []
    num_notes_post_song = []
    
    for i_song, song_info in df_songs.iterrows():
        t_song_start = song_info['start']
        t_song_end = song_info['end']

        t_pre_window_start = t_song_start - context_duration
        t_pre_window_end = t_song_start
        notes_in_pre_window = df_notes[
            (df_notes['start']>t_pre_window_start) &
            (df_notes['end']<t_pre_window_end)]
        
        t_post_window_start = t_song_end
        t_post_window_end = t_song_end + context_duration
        notes_in_post_window = df_notes[
            (df_notes['start']>t_post_window_start) &
            (df_notes['end']<t_post_window_end)]
        
        num_notes_pre_song.append(len(notes_in_pre_window))
        num_notes_post_song.append(len(notes_in_post_window))

    df_songs['num_notes_pre_song'] = num_notes_pre_song
    df_songs['num_notes_post_song'] = num_notes_post_song

    return df_songs

def assign_bout_duration_to_notes(df_notes, vocal_bouts):
    """
    Given the original notes and the computed vocal bouts, assign each note a new column 
    'bout_duration' such that:
      - If the note is part of a vocal bout (i.e. its start and end times fall within the 
        bout boundaries), then 'bout_duration' is the overall bout duration.
      - Otherwise, if the note is isolated, then 'bout_duration' is just the note's own duration.
    
    Parameters:
      df_notes (pd.DataFrame): DataFrame containing individual notes with at least 'start' and 'end' columns.
      vocal_bouts (pd.DataFrame): DataFrame of vocal bouts (merged intervals) with columns 
                                  'start', 'end', and 'duration' (i.e. bout duration).
    
    Returns:
      pd.DataFrame: A copy of df_notes with an added 'bout_duration' column.
    """
    # Make a copy so we do not modify the original data
    df_notes = df_notes.copy()
    
    # First, initialize bout_duration with each note's own duration.
    df_notes['bout_duration'] = df_notes['end'] - df_notes['start']
    
    # For each bout in vocal_bouts, update the bout_duration for notes that fall entirely within the bout.
    for _, bout in vocal_bouts.iterrows():
        # We assume vocal_bouts already has a 'duration' column. 
        # If not, you can compute it as: bout_duration = bout['end'] - bout['start']
        bout_duration = bout['duration']  
        
        # Create a boolean mask: note is in the bout if its start and end are within the bout boundaries.
        mask = (df_notes['start'] >= bout['start']) & (df_notes['end'] <= bout['end'])
        df_notes.loc[mask, 'bout_duration'] = bout_duration
    
    return df_notes

def characterize_song_context(df_notes, df_songs, gap_threshold):
    """
    For each song in df_songs, extend its context (song bout) by looking for note bouts
    that overlap the song. If multiple bouts overlap, the song's context is the union 
    (min start and max end) of all overlapping bouts.
    
    In addition, count how many note events (from df_notes) in the song bout occur before 
    the song and how many occur after the song.
    
    Parameters:
    - df_notes (pd.DataFrame): DataFrame of note events with 'start' and 'end' columns.
    - df_songs (pd.DataFrame): DataFrame of songs with 'start' and 'end' columns.
    - gap_threshold (float): Maximum allowed gap between consecutive notes to merge them.
    
    Returns:
    - df_songs (pd.DataFrame): A copy of df_songs with four new columns:
         'song_bout_start', 'song_bout_end', 'song_bout_notes_before', and 'song_bout_notes_after'
    """
    # Build bouts from notes.
    vocal_bouts, isolated_notes = find_vocal_bouts(df_notes, gap_threshold)
    
    # Combine both merged bouts and isolated notes.
    bouts = pd.concat(
        [
            vocal_bouts[['start', 'end']],
            isolated_notes[['start', 'end']]
        ],
        ignore_index=True
    )
    bouts = bouts.sort_values('start').reset_index(drop=True)
    
    # Prepare lists for song bout information and note counts.
    bout_starts = []
    bout_ends = []
    notes_before_counts = []
    notes_after_counts = []
    t_before = []
    t_after = []
    
    # Iterate over each song.
    for _, song in df_songs.iterrows():
        # Find all bouts that overlap the song.
        matching_bouts = bouts[(bouts['start'] <= song['end']) & (bouts['end'] >= song['start'])]
        
        if not matching_bouts.empty:
            # The song's bout is the union of all matching bouts.
            song_context_start = matching_bouts['start'].min()
            song_context_end = matching_bouts['end'].max()
            bout_starts.append(song_context_start)
            bout_ends.append(song_context_end)
            
            # Find all note events (from df_notes) that contributed to this bout.
            bout_notes = df_notes[(df_notes['start'] >= song_context_start) & (df_notes['end'] <= song_context_end)]
            
            # Count notes that finish on or before the song starts (i.e. "before" the song).
            notes_before = bout_notes[bout_notes['end'] < song['start']].shape[0]
            # Count notes that start on or after the song ends (i.e. "after" the song).
            notes_after = bout_notes[bout_notes['start'] > song['end']].shape[0]
            
            notes_before_counts.append(notes_before)
            notes_after_counts.append(notes_after)

            t_before.append(song['start'] - song_context_start)
            t_after.append(song_context_end - song['end'])
        else:
            # No overlapping bout: set bout boundaries to NaN and counts to 0.
            bout_starts.append(np.nan)
            bout_ends.append(np.nan)
            notes_before_counts.append(0)
            notes_after_counts.append(0)
            t_before.append(np.nan)
            t_after.append(np.nan)
    
    # Add new columns to the songs DataFrame.
    df_songs = df_songs.copy()
    df_songs['song_bout_start'] = bout_starts
    df_songs['song_bout_end'] = bout_ends
    df_songs['song_bout_notes_before'] = notes_before_counts
    df_songs['song_bout_notes_after'] = notes_after_counts
    df_songs['song_bout_t_before'] = t_before
    df_songs['song_bout_t_after'] = t_after

    
    return df_songs

def is_not_in_intervals(t, intervals):
    return not intervals.contains(t).any()

def find_intervals(t, gap_thr=0.01, min_dur=0.01, max_dur=0.25):
    """
    given a numpy array of time t, find the intervals that contain these t with a minimum gap threshold
    
    """
    if len(t) == 0:
        return []
    
    t_sorted = np.sort(t)  # Ensure t is sorted
    
    # Find where the difference between consecutive elements is >= gap_threshold
    gaps = np.where(np.diff(t_sorted) > gap_thr)[0]
    
    # Identify the start and end of each interval
    interval_starts = np.insert(gaps + 1, 0, 0)
    interval_ends = np.append(gaps, len(t_sorted) - 1)
    
    # Use starts and ends to create intervals
    intervals = np.vstack((t_sorted[interval_starts], t_sorted[interval_ends])).T
    
    # Calculate durations and create a mask for filtering
    durations = intervals[:, 1] - intervals[:, 0]
    mask = (durations >= min_dur) & (durations <= max_dur)
    
    # Apply the mask to filter intervals
    filtered_intervals = intervals[mask]
    
    return filtered_intervals

Note: you may need to restart the kernel to use updated packages.


In [11]:
# import internal libraries
from utils import (
    compute_1d_kde,
    compute_2d_kde,
    find_threshold_between_peaks,
    draw_ellipse,
)

from events import (
    compute_event_features,
    compute_window_features,
    merge_stereotypy_windows,
    characterize_segments,
    classify_segments,
    classify_notes_by_temporal_patterning,
    classify_notes_by_amplitude,
    find_vocal_bouts,
    analyze_song_patterning,
    get_song_social_context,
    assign_bout_duration_to_notes,
    characterize_song_context,
)

ModuleNotFoundError: No module named 'utils'

## Parameters

In [14]:
path_to_nas = '/Users/kdawson/Documents/GitHub/zheng2025/original_paper/telc_data/pag_telc'

# identify loud vs soft notes
amp_high_threshold = -5 # based on solosocial-social, separating the two modes
amp_song_threshold = -55

# identify local windows of stereotyped sequences
min_window_size = 7
window_max_log_isi_threshold = -1 # based on solosocial-social, separating local stereotypy vs not
window_rmse_threshold = 0.15 # based on solosocial-social, separating local stereotypy vs not

# merge local windows into long segments
segment_max_time_gap = 0.5
segment_max_index_gap = 5

# load the sklearn models for segment features
with open('segment_features_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
# Load the PCA
with open('segment_features_pca.pkl', 'rb') as f:
    pca = pickle.load(f)
# Load the GMM
with open('segment_features_gmm.pkl', 'rb') as f:
    gmm = pickle.load(f)

cluster_mapping = {0: 'S2', 1: 'S3', 2: 'S1'}

# color palettes for plotting
color_palette_patterning = {
    'S1': 'mediumturquoise',
    'S2': 'royalblue',
    'S3': 'blueviolet',
    'U' : 'orchid',
}

# final clusters and mapping
final_type_mapping = {'S1': 'song', 'S2': 'proto-song', 'S3': 'USV', 'U': 'USV'}
final_color_palette_patterning = {
    'song': 'mediumturquoise',
    'proto-song': 'royalblue',
    'USV': 'purple',
}

song_final_note_occupancy = 0.5

FileNotFoundError: [Errno 2] No such file or directory: 'segment_features_scaler.pkl'

In [ ]:
# plotting
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 6

## Load session info

In [ ]:
sessions_info_filepath = os.path.join(path_to_nas, 'pag_telc_sessions.csv')
sessions_info = pd.read_csv(sessions_info_filepath, header=0)
sessions_info = sessions_info[(sessions_info['condition']=='pre')|(sessions_info['condition']=='post')].reset_index(drop=True)
sessions_info

In [ ]:
sessions_curated_hours_info_filepath = os.path.join(path_to_nas, 'usv_calls', '251007_dryad_upload', 'pag_telc', 'pag_telc_curation.csv')
sessions_curated_hours_info = pd.read_csv(sessions_curated_hours_info_filepath, header=0)
sessions_curated_hours_info

In [ ]:
# make color palette
sorted_animal_ids = sorted(sessions_info['mouse_1'].unique())
palette = sns.color_palette(n_colors=len(sorted_animal_ids))
palette_dict = dict(zip(sorted_animal_ids, palette))

In [ ]:
all_sessions = []
for _, session_info in sessions_info.iterrows():

    session = {}

    session_id = session_info['session']
    project_dir = session_info['basepath']
    condition = session_info['condition']
    mouse_1 = session_info['mouse_1']
    mouse_2 = session_info['mouse_2']
    anchor_file_idx = int(session_info['anchor_file_idx'])
    num_vocal_days = int(session_info['num_vocal_days'])
    num_post_days = session_info['num_post_days']
    session_basepath = os.path.join(path_to_nas, 'usv_calls', '251007_dryad_upload', 'pag_telc', session_id)
    print(f'Processing session: {session_basepath}')

    if condition == 'pre':
        audio_dir = 'process_audio'
    else:
        audio_dir = 'original_audio'
    base_dir = Path(f'{session_basepath}/{audio_dir}/all_notes')
    file_indices = sorted([p.name for p in base_dir.iterdir() if p.is_dir()])
    print(f'Num hours: {len(file_indices)}')

    # find curated hours
    session_curated_hours = sessions_curated_hours_info[sessions_curated_hours_info['session']==session_id]
    print(f'Number of curated hours in this session: {len(session_curated_hours)}')
    curated_hours_file_indices = session_curated_hours['hr'].values

    # fill metadata info
    session['session_id'] = session_id
    session['project_dir'] = project_dir
    session['session_basepath'] = session_basepath
    session['condition'] = condition
    session['anchor_file_idx'] = anchor_file_idx
    session['num_post_days'] = num_post_days
    session['num_vocal_days'] = num_vocal_days
    session['mouse_1'] = mouse_1
    session['mouse_2'] = mouse_2
    session['num_files'] = len(file_indices)
    session['num_curated_files'] = len(curated_hours_file_indices)

    # all hours

    # for each hour, get number of calls
    hours_info = []
    hours_all_notes = []
    hours_l_notes = []
    hours_r_notes = []
    for i_file, file_idx in enumerate(file_indices):
        
        # # get start time in world time
        # l_audio_filepath = glob.glob(os.path.join(session_basepath, 'original_audio', 'left', f'*_{file_idx}.wav'))[0]
        # audio_filename = os.path.splitext(os.path.basename(l_audio_filepath))[0]
        # audio_filename_timestamp = pd.to_datetime(audio_filename.split('_')[0][1:], format='%y%m%d%H%M%S')
        
        # get all notes
        if session['condition'] == 'pre':
            all_notes_dir = os.path.join(session_basepath, 'process_audio', 'all_notes', file_idx)
        else:
            all_notes_dir = os.path.join(session_basepath, 'original_audio', 'all_notes', file_idx)
            
        if os.path.exists((os.path.join(all_notes_dir, 'all_notes_corrected_features.csv'))):
            all_notes_filepath = os.path.join(all_notes_dir, 'all_notes_corrected_features.csv')
        elif os.path.exists((os.path.join(all_notes_dir, 'all_notes_corrected.csv'))):
            all_notes_filepath = os.path.join(all_notes_dir, 'all_notes_corrected.csv')
        elif os.path.exists((os.path.join(all_notes_dir, 'all_notes.csv'))):
            all_notes_filepath = os.path.join(all_notes_dir, 'all_notes.csv')
        else:
            print(f'Cannot find any all_notes feature file, skipping session: {session_id}, file_idx: {file_idx}')
            continue
        # print('reading file', all_notes_filepath)

        # load notes
        all_notes = pd.read_csv(all_notes_filepath, header=0, index_col=0)
        all_notes = all_notes.drop_duplicates(subset=['start', 'end'], keep='last')
        all_notes = all_notes.sort_values(by=['start', 'end']).reset_index(drop=True)

        all_notes['session_id'] = session['session_id']
        all_notes['file_idx'] = file_idx
        l_notes = all_notes[all_notes['code']==0].copy().reset_index(drop=True)
        r_notes = all_notes[all_notes['code']==1].copy().reset_index(drop=True)
        l_notes['mouse_id'] = session['mouse_1']
        l_notes['mouse_side'] = 'mouse_1'
        r_notes['mouse_id'] = session['mouse_2']
        r_notes['mouse_side'] = 'mouse_2'

        # save the hour's notes
        hours_all_notes.append(all_notes)
        hours_l_notes.append(l_notes)
        hours_r_notes.append(r_notes)

        # save the hour's info
        hours_info.append({
            'i_file': i_file,
            'file_idx': file_idx,
            # 'file_timestamp': audio_filename_timestamp,
            'all_notes_filepath': all_notes_filepath,
            'l_num_notes': len(l_notes),
            'r_num_notes': len(r_notes),
        })

    df_hours_info = pd.DataFrame(hours_info)
    session['hours_info'] = df_hours_info
    # session['session_start_timestamp'] = df_hours_info['file_timestamp'].min()
    # session['session_reference_day'] = session['session_start_timestamp'].date()
    session['hours_all_notes'] = hours_all_notes
    session['hours_l_notes'] = hours_l_notes
    session['hours_r_notes'] = hours_r_notes

    # curated hours
    curated_hours_info = []
    curated_hours_all_notes = []
    curated_hours_l_notes = []
    curated_hours_r_notes = []
    for _, file_idx in enumerate(curated_hours_file_indices):

        # # get start time in world time
        # l_audio_filepath = glob.glob(os.path.join(session_basepath, 'original_audio', 'left', f'*_{file_idx:07d}.wav'))[0]
        # audio_filename = os.path.splitext(os.path.basename(l_audio_filepath))[0]
        # audio_filename_timestamp = pd.to_datetime(audio_filename.split('_')[0][1:], format='%y%m%d%H%M%S')

        # get all notes
        if session['condition'] == 'pre':
            all_notes_dir = os.path.join(session_basepath, 'process_audio', 'all_notes', f'{file_idx:07d}')
        else:
            all_notes_dir = os.path.join(session_basepath, 'original_audio', 'all_notes', f'{file_idx:07d}')
            
        if os.path.exists((os.path.join(all_notes_dir, 'all_notes_corrected_features.csv'))):
            all_notes_filepath = os.path.join(all_notes_dir, 'all_notes_corrected_features.csv')
        else:
            print(f'Cannot find any curated all_notes feature file, skipping session: {session_id}, file_idx: {file_idx:07d}')
            continue
        # print('reading file', all_notes_filepath)

        # load notes
        all_notes = pd.read_csv(all_notes_filepath, header=0, index_col=0)
        all_notes = all_notes.drop_duplicates(subset=['start', 'end'], keep='last')
        all_notes = all_notes.sort_values(by=['start', 'end']).reset_index(drop=True)

        all_notes['session_id'] = session['session_id']
        all_notes['file_idx'] = file_idx
        l_notes = all_notes[all_notes['code']==0].copy().reset_index(drop=True)
        r_notes = all_notes[all_notes['code']==1].copy().reset_index(drop=True)
        l_notes['mouse_id'] = session['mouse_1']
        l_notes['mouse_side'] = 'mouse_1'
        r_notes['mouse_id'] = session['mouse_2']
        r_notes['mouse_side'] = 'mouse_2'

        # save the hour's notes
        curated_hours_all_notes.append(all_notes)
        curated_hours_l_notes.append(l_notes)
        curated_hours_r_notes.append(r_notes)

        # save the hour's info
        curated_hours_info.append({
            'i_file': np.where(np.array(file_indices).astype(int)==file_idx)[0][0],
            'file_idx': file_idx,
            # 'file_timestamp': audio_filename_timestamp,
            'all_notes_filepath': all_notes_filepath,
            'l_num_notes': len(l_notes),
            'r_num_notes': len(r_notes),
        })

    df_curated_hours_info = pd.DataFrame(curated_hours_info)
    session['curated_hours_info'] = df_curated_hours_info
    session['curated_hours_all_notes'] = curated_hours_all_notes
    session['curated_hours_l_notes'] = curated_hours_l_notes
    session['curated_hours_r_notes'] = curated_hours_r_notes

    # save the output
    all_sessions.append(session)

## Overview pre vs post

In [ ]:
# select overview hours
for session in all_sessions:

    overview_hours_info = []
    overview_hours_all_notes = []
    overview_hours_l_notes = []
    overview_hours_r_notes = []

    session_id = session['session_id']
    mouse_1 = session['mouse_1']
    mouse_2 = session['mouse_2']
    condition = session['condition']

    print(f'Processing session: {session_id}; mouse_1: {mouse_1}; condition: {condition}')
    # Process hours_info
    hours_info = session['hours_info'].copy()
    # hours_info['file_timestamp'] = pd.to_datetime(hours_info['file_timestamp'])
    # reference_day = pd.Timestamp(session['session_reference_day'])
    # hours_info['hour_offset'] = (hours_info['file_timestamp'] - reference_day).dt.total_seconds() / 3600.0
    
    # Get the anchor file index and look up the corresponding row
    anchor_file_idx = session["anchor_file_idx"]
    # # Get the anchor file index and look up the corresponding row
    # anchor_file_idx = session["anchor_file_idx"]
    # anchor_row = hours_info[hours_info['file_idx'].astype(int) == anchor_file_idx]
    # if anchor_row.empty:
    #     print(f"Anchor file idx {anchor_file_idx} not found for session {session['session_id']}")
    #     continue
    # anchor_hour_offset = anchor_row.iloc[0]['hour_offset']
    # print(f'anchor_file_idx: {anchor_file_idx}; time offset: {anchor_hour_offset:.2f}')
    
    anchor_row = hours_info[hours_info['file_idx'].astype(int) == anchor_file_idx]
    # anchor_row_index = anchor_row.index[0]
    # if anchor_row.empty:
    #     print(f"Anchor file idx {anchor_file_idx} not found for session {session['session_id']}")
    #     continue
    # print(f'anchor i_file: {anchor_row_index}')

    # For pre sessions, record day 1 only using the anchor hour
    if condition == 'pre':
        if anchor_row.empty:
            print(f"Anchor file idx {anchor_file_idx} not found for session {session['session_id']}")
            continue
        else:
            anchor_row_index = anchor_row.index[0]
            print(f'pre; file: {hours_info.iloc[anchor_row_index]["all_notes_filepath"]}')
            overview_hours_info.append(hours_info.iloc[anchor_row_index])
            overview_hours_all_notes.append(session['hours_all_notes'][anchor_row_index])
            overview_hours_l_notes.append(session['hours_l_notes'][anchor_row_index])
            overview_hours_r_notes.append(session['hours_r_notes'][anchor_row_index])
    
    # For post sessions, record data for 5 days at 24-hour increments starting at the anchor hour.
    else:
        for day in range(1, int(session['num_post_days']) + 1):
            target_file_index = anchor_file_idx + 24 * (day - 1)
            target_row = hours_info[hours_info['file_idx'].astype(int) == target_file_index]
            if target_row.empty:
                print(f"Target file idx {target_file_index} not found for session {session['session_id']} on day {day}")
                continue
            else:
                row_index = target_row.index[0]
                print(f'post day: {day}; file: {hours_info.iloc[row_index]["all_notes_filepath"]}')
                # target_time = anchor_hour_offset + 24 * day
                # row_index = (hours_info['hour_offset'] - target_time).abs().idxmin()
                # print(f'post day: {day}; hour offset: {hours_info.iloc[row_index]["hour_offset"] - anchor_hour_offset:.2f}; file: {hours_info.iloc[row_index]["all_notes_filepath"]}')
                overview_hours_info.append(hours_info.iloc[row_index])
                overview_hours_all_notes.append(session['hours_all_notes'][row_index])
                overview_hours_l_notes.append(session['hours_l_notes'][row_index])
                overview_hours_r_notes.append(session['hours_r_notes'][row_index])
    
    df_overview_hours_info = pd.DataFrame(overview_hours_info).reset_index(drop=True)
    session['overview_hours_info'] = df_overview_hours_info
    session['overview_hours_all_notes'] = overview_hours_all_notes
    session['overview_hours_l_notes'] = overview_hours_l_notes
    session['overview_hours_r_notes'] = overview_hours_r_notes

In [ ]:
# analyze temporal patterning
for session in all_sessions:
    print(f'Analyzing session: {session["session_id"]}')
    print(f'Num overview hours: {len(session["overview_hours_l_notes"])}')

    # Process left notes
    processed_overview_left_notes = []
    processed_overview_left_windows = []
    processed_overview_left_filtered_windows = []
    processed_overview_left_segments = []
    processed_overview_left_songs = []

    for hour_df in session['overview_hours_l_notes']:
        print(f'Num left notes: {len(hour_df)}')
        # compute temporal features - note level
        df_events = compute_event_features(hour_df)
        # compute temporal features - fixed size window
        df_windows = compute_window_features(df_events, min_window_size=min_window_size)
        # join fixed size windows into long segments
        df_filtered_windows, df_segments = merge_stereotypy_windows(df_windows,
                                                                    df_events,
                                                                    max_log_interval_threshold=window_max_log_isi_threshold,
                                                                    rmse_threshold=window_rmse_threshold,
                                                                    max_gap=segment_max_time_gap,
                                                                    max_index_gap=segment_max_index_gap)
        # characterize segments
        df_segments = characterize_segments(df_events, 
                                            df_segments, 
                                            ransac_min_samples=min_window_size,
                                            ransac_normalized_residual_threshold=window_rmse_threshold,)
        # classify segments
        df_segments = classify_segments(df_segments, scaler, pca, gmm, cluster_mapping=cluster_mapping)
        # classify notes by temporal patterning based on the segment it's in
        df_events = classify_notes_by_temporal_patterning(df_segments, df_events)
        # # classify notes by loudness
        # df_events = classify_notes_by_amplitude(df_events, amp_high_threshold)
        # consolidate vocalizations into final types
        df_events['type_final'] = df_events['type_patterning'].map(final_type_mapping)

        # analyze songs defined by the linear backbone of the song segment
        if df_segments.empty:
            df_songs = pd.DataFrame()
        else:
            df_song_segments = df_segments[df_segments['cluster_label']=='S1']
            # df_song_segments = df_segments[(df_segments['cluster_label']=='S1') | (df_segments['cluster_label']=='S2')]
            df_songs = analyze_song_patterning(df_song_segments, df_events, song_final_note_occupancy=song_final_note_occupancy)
        
        processed_overview_left_notes.append(df_events)
        processed_overview_left_windows.append(df_windows)
        processed_overview_left_filtered_windows.append(df_filtered_windows)
        processed_overview_left_segments.append(df_segments)
        processed_overview_left_songs.append(df_songs)

    session['overview_hours_l_notes'] = processed_overview_left_notes
    session['overview_hours_l_windows'] = processed_overview_left_windows
    session['overview_hours_l_filtered_windows'] = processed_overview_left_filtered_windows
    session['overview_hours_l_segments'] = processed_overview_left_segments
    session['overview_hours_l_songs'] = processed_overview_left_songs

In [ ]:
overview_hours = []

# Loop through sessions to collect data points
for session in all_sessions:

    session_id = session['session_id']
    mouse_1 = session['mouse_1']
    mouse_2 = session['mouse_2']
    condition = session['condition']

    print(f'Processing session: {session_id}; mouse_1: {mouse_1}; condition: {condition}')

    # Process hours_info
    hours_info = session['hours_info'].copy()
    # hours_info['file_timestamp'] = pd.to_datetime(hours_info['file_timestamp'])
    # reference_day = pd.Timestamp(session['session_reference_day'])
    # hours_info['hour_offset'] = (hours_info['file_timestamp'] - reference_day).dt.total_seconds() / 3600.0

    # Get the anchor file index and look up the corresponding row
    anchor_file_idx = session["anchor_file_idx"]
    # # Get the anchor file index and look up the corresponding row
    # anchor_file_idx = session["anchor_file_idx"]
    anchor_row = hours_info[hours_info['file_idx'].astype(int) == anchor_file_idx]
    # if anchor_row.empty:
    #     print(f"Anchor file idx {anchor_file_idx} not found for session {session['session_id']}")
    #     continue
    # anchor_hour_offset = anchor_row.iloc[0]['hour_offset']
    # print(f'anchor_file_idx: {anchor_file_idx}; time offset: {anchor_hour_offset:.2f}')

    df_overview_hours_info = session['overview_hours_info'].copy()

    # For pre sessions, record day 1 only using the anchor hour
    if condition == 'pre':
        row_index = df_overview_hours_info.iloc[0]['i_file']
        print(f'pre; row index: {row_index}; file: {hours_info.iloc[row_index]["all_notes_filepath"]}')
        l_num_notes = len(session['overview_hours_l_notes'][0])
        r_num_notes = len(session['overview_hours_r_notes'][0])
        l_notes = session['overview_hours_l_notes'][0]
        l_songs = session['overview_hours_l_songs'][0]
        l_num_usvs = len(l_notes[l_notes['type_final']=='USV'])
        l_num_songs = len(l_songs)
        # hour_offset = hours_info.iloc[row_index]['hour_offset']
        overview_hours.append({
            'session_id': session_id,
            'condition': condition,
            'mouse_1': mouse_1,
            'mouse_2': mouse_2,
            'day': 0,
            # 'hour_offset': hour_offset,
            'all_notes_filepath': hours_info.iloc[row_index]['all_notes_filepath'],
            'l_num_notes': l_num_notes,
            'r_num_notes': r_num_notes,
            'l_num_usvs': l_num_usvs,
            'l_num_songs': l_num_songs,
        })

    # For post sessions, record data for 5 days at 24-hour increments starting at the anchor hour.
    else:
        for i_day, overview_hour in df_overview_hours_info.iterrows():
            row_index = overview_hour['i_file']
            print(f'post day: {i_day+1}; file: {hours_info.iloc[row_index]["all_notes_filepath"]}')
            # hour_offset = hours_info.iloc[row_index]['hour_offset']
            l_num_notes = len(session['overview_hours_l_notes'][i_day])
            r_num_notes = len(session['overview_hours_r_notes'][i_day])
            l_notes = session['overview_hours_l_notes'][i_day]
            l_songs = session['overview_hours_l_songs'][i_day]
            l_num_usvs = len(l_notes[l_notes['type_final']=='USV'])
            l_num_songs = len(l_songs)
            overview_hours.append({
                'session_id': session_id,
                'condition': condition,
                'mouse_1': mouse_1,
                'mouse_2': mouse_2,
                'day': i_day+1,
                # 'hour_offset': hour_offset,
                'all_notes_filepath': hours_info.iloc[row_index]['all_notes_filepath'],
                'l_num_notes': l_num_notes,
                'r_num_notes': r_num_notes,
                'l_num_usvs': l_num_usvs,
                'l_num_songs': l_num_songs,
            })

df_overview_hours = pd.DataFrame(overview_hours)

In [ ]:
# let's first do a quick pre-post - usv
# pre-first day, post-last day
fig, ax = plt.subplots(figsize=(1.5, 1.75))

# get each mouse
df_mouse_pre_post_summary = []
for mouse_1 in df_overview_hours['mouse_1'].unique():
    # get pre and post
    pre = df_overview_hours[(df_overview_hours['condition']=='pre')&(df_overview_hours['mouse_1']==mouse_1)].iloc[0]
    post = df_overview_hours[(df_overview_hours['condition']=='post')&(df_overview_hours['mouse_1']==mouse_1)].iloc[-1]
    df_mouse_pre_post_summary.extend([pre, post])
df_mouse_pre_post_summary = pd.DataFrame(df_mouse_pre_post_summary)

# plot individual mouse
for mouse_1 in df_mouse_pre_post_summary['mouse_1'].unique():
    df_mouse = df_mouse_pre_post_summary[df_mouse_pre_post_summary['mouse_1']==mouse_1]
    ax.plot([0, 1], df_mouse['l_num_usvs'], marker='o', markersize=4, label=mouse_1, color=palette_dict[mouse_1])

# plot mean +- sem
pre_l_mean = df_mouse_pre_post_summary[df_mouse_pre_post_summary['condition']=='pre']['l_num_usvs'].mean()
pre_l_sem = df_mouse_pre_post_summary[df_mouse_pre_post_summary['condition']=='pre']['l_num_usvs'].sem()
post_l_mean = df_mouse_pre_post_summary[df_mouse_pre_post_summary['condition']=='post']['l_num_usvs'].mean()
post_l_sem = df_mouse_pre_post_summary[df_mouse_pre_post_summary['condition']=='post']['l_num_usvs'].sem()

ax.errorbar([0, 1], [pre_l_mean, post_l_mean], yerr=[pre_l_sem, post_l_sem], 
            fmt='-o', markersize=4, capsize=4, color='k')

# annotate plot
ax.legend(loc='upper right', fontsize='x-small')
ax.set_ylim([0, 7000])
ax.set_xticks([0, 1], ['Pre', 'Post'])
ax.set_xlim([-0.5, 1.5])

ax.set_ylabel('USVs / hour')
plt.tight_layout()

# plt.show()

In [ ]:
# let's first do a quick pre-post - song
# pre-first day, post-last day
fig, ax = plt.subplots(figsize=(1.5, 1.75))

# get each mouse
df_mouse_pre_post_summary = []
for mouse_1 in df_overview_hours['mouse_1'].unique():
    # get pre and post
    pre = df_overview_hours[(df_overview_hours['condition']=='pre')&(df_overview_hours['mouse_1']==mouse_1)].iloc[0]
    post = df_overview_hours[(df_overview_hours['condition']=='post')&(df_overview_hours['mouse_1']==mouse_1)].iloc[-1]
    df_mouse_pre_post_summary.extend([pre, post])
df_mouse_pre_post_summary = pd.DataFrame(df_mouse_pre_post_summary)

# plot individual mouse
for mouse_1 in df_mouse_pre_post_summary['mouse_1'].unique():
    df_mouse = df_mouse_pre_post_summary[df_mouse_pre_post_summary['mouse_1']==mouse_1]
    ax.plot([0, 1], df_mouse['l_num_songs'], marker='o', markersize=4, label=mouse_1, color=palette_dict[mouse_1])

# plot mean +- sem
pre_l_mean = df_mouse_pre_post_summary[df_mouse_pre_post_summary['condition']=='pre']['l_num_songs'].mean()
pre_l_sem = df_mouse_pre_post_summary[df_mouse_pre_post_summary['condition']=='pre']['l_num_songs'].sem()
post_l_mean = df_mouse_pre_post_summary[df_mouse_pre_post_summary['condition']=='post']['l_num_songs'].mean()
post_l_sem = df_mouse_pre_post_summary[df_mouse_pre_post_summary['condition']=='post']['l_num_songs'].sem()

print(f'pre song: {pre_l_mean:.2f} +- {pre_l_sem:.2f}')
print(f'post song: {post_l_mean:.2f} +- {post_l_sem:.2f}')

ax.errorbar([0, 1], [pre_l_mean, post_l_mean], yerr=[pre_l_sem, post_l_sem], 
            fmt='-o', markersize=4, capsize=4, color='k')

# annotate plot
ax.legend(loc='upper right', fontsize='x-small')
ax.set_ylim([0, 50])
ax.set_xticks([0, 1], ['Pre', 'Post'])
ax.set_xlim([-0.5, 1.5])

ax.set_ylabel('Songs / hour')
plt.tight_layout()

plt.show()

## Focus on curated hours, analyze how vocalizations are perturbed before going mute

In [ ]:
# analyze temporal patterning
for session in all_sessions:
    print(f'Analyzing session: {session["session_id"]}')
    print(f'Num curated hours: {len(session["curated_hours_l_notes"])}')

    # Process left notes
    processed_curated_left_notes = []
    processed_curated_left_windows = []
    processed_curated_left_filtered_windows = []
    processed_curated_left_segments = []
    processed_curated_left_songs = []

    # store the bouts
    processed_curated_left_vocal_bouts_song = []
    processed_curated_left_vocal_bouts_proto_song = []
    processed_curated_left_vocal_bouts_usv = []
    processed_curated_left_isolated_notes_song = []
    processed_curated_left_isolated_notes_proto_song = []
    processed_curated_left_isolated_notes_usv = []

    for hour_df in session['curated_hours_l_notes']:
        print(f'Num left notes: {len(hour_df)}')
        # compute temporal features - note level
        df_events = compute_event_features(hour_df)
        # compute temporal features - fixed size window
        df_windows = compute_window_features(df_events, min_window_size=min_window_size)
        # join fixed size windows into long segments
        df_filtered_windows, df_segments = merge_stereotypy_windows(df_windows,
                                                                    df_events,
                                                                    max_log_interval_threshold=window_max_log_isi_threshold,
                                                                    rmse_threshold=window_rmse_threshold,
                                                                    max_gap=segment_max_time_gap,
                                                                    max_index_gap=segment_max_index_gap)
        # characterize segments
        df_segments = characterize_segments(df_events, 
                                            df_segments, 
                                            ransac_min_samples=min_window_size,
                                            ransac_normalized_residual_threshold=window_rmse_threshold,)
        # classify segments
        df_segments = classify_segments(df_segments, scaler, pca, gmm, cluster_mapping=cluster_mapping)
        # classify notes by temporal patterning based on the segment it's in
        df_events = classify_notes_by_temporal_patterning(df_segments, df_events)
        # classify notes by loudness
        df_events = classify_notes_by_amplitude(df_events, amp_high_threshold)
        # consolidate vocalizations into final types
        df_events['type_final'] = df_events['type_patterning'].map(final_type_mapping)

        # analyze songs defined by the linear backbone of the song segment
        if df_segments.empty:
            df_songs = pd.DataFrame()
        else:
            df_song_segments = df_segments[df_segments['cluster_label']=='S1']
            # df_song_segments = df_segments[(df_segments['cluster_label']=='S1') | (df_segments['cluster_label']=='S2')]
            df_songs = analyze_song_patterning(df_song_segments, df_events, song_final_note_occupancy=song_final_note_occupancy)

        # bout analysis
        vocal_bouts_song, isolated_notes_song = find_vocal_bouts(df_events[df_events['type_final']=='song'], gap_threshold=0.1)
        vocal_bouts_proto_song, isolated_notes_proto_song = find_vocal_bouts(df_events[df_events['type_final']=='proto-song'], gap_threshold=0.1)
        vocal_bouts_usv, isolated_notes_usv = find_vocal_bouts(df_events[df_events['type_final']=='USV'], gap_threshold=0.1)
        # store the bouts
        processed_curated_left_vocal_bouts_song.append(vocal_bouts_song)
        processed_curated_left_vocal_bouts_proto_song.append(vocal_bouts_proto_song)
        processed_curated_left_vocal_bouts_usv.append(vocal_bouts_usv)
        processed_curated_left_isolated_notes_song.append(isolated_notes_song)
        processed_curated_left_isolated_notes_proto_song.append(isolated_notes_proto_song)
        processed_curated_left_isolated_notes_usv.append(isolated_notes_usv)
        
        processed_curated_left_notes.append(df_events)
        processed_curated_left_windows.append(df_windows)
        processed_curated_left_filtered_windows.append(df_filtered_windows)
        processed_curated_left_segments.append(df_segments)
        processed_curated_left_songs.append(df_songs)

    session['curated_hours_l_notes'] = processed_curated_left_notes
    session['curated_hours_l_windows'] = processed_curated_left_windows
    session['curated_hours_l_filtered_windows'] = processed_curated_left_filtered_windows
    session['curated_hours_l_segments'] = processed_curated_left_segments
    session['curated_hours_l_songs'] = processed_curated_left_songs
    # store the bouts
    session['curated_hours_l_vocal_bouts_song'] = processed_curated_left_vocal_bouts_song
    session['curated_hours_l_vocal_bouts_proto_song'] = processed_curated_left_vocal_bouts_proto_song
    session['curated_hours_l_vocal_bouts_usv'] = processed_curated_left_vocal_bouts_usv
    session['curated_hours_l_isolated_notes_song'] = processed_curated_left_isolated_notes_song
    session['curated_hours_l_isolated_notes_proto_song'] = processed_curated_left_isolated_notes_proto_song
    session['curated_hours_l_isolated_notes_usv'] = processed_curated_left_isolated_notes_usv

In [ ]:
# designate post-early vs post-late
for session in all_sessions:
    if session['condition'] == 'pre':
        continue
    df = session['curated_hours_info']
    total_notes = df['l_num_notes'].sum()
    cut_point = total_notes / 2
    # Compute the cumulative sum of l_num_notes
    cumsum = df['l_num_notes'].cumsum()
    crossing_index = cumsum[cumsum >= cut_point].index[0]
    # Create a new column 'period' based on whether the cumulative sum is <= cut_point
    df['post_period'] = ['early' if idx <= crossing_index else 'late' for idx in df.index]


In [ ]:
# collect all notes
all_curated_hours_notes = []

for session in all_sessions:

    session_id = session['session_id']
    mouse_1 = session['mouse_1']
    mouse_2 = session['mouse_2']
    condition = session['condition']

    curated_hours_info = session['curated_hours_info'].copy()

    print(f'Processing session: {session_id}; mouse_1: {mouse_1}; condition: {condition}')

    for i_hour, curated_hour in curated_hours_info.iterrows():

        df_notes = session['curated_hours_l_notes'][i_hour]

        # add metadata
        df_notes['mouse_id'] = mouse_1
        df_notes['condition'] = condition
        df_notes['session_id'] = session_id
        df_notes['file_idx'] = curated_hour['file_idx']
        if condition == 'pre':
            df_notes['full_condition'] = 'pre'
        else:
            post_period = curated_hour['post_period']
            print(f'file_idx: {curated_hour["file_idx"]}; post_period: {post_period}')
            df_notes['full_condition'] = f'post_{post_period}'

        # save
        all_curated_hours_notes.append(df_notes)

df_all_curated_hours_notes = pd.concat(all_curated_hours_notes, ignore_index=True)

In [ ]:
# collect all local windows
all_curated_hours_windows = []

for session in all_sessions:

    session_id = session['session_id']
    mouse_1 = session['mouse_1']
    mouse_2 = session['mouse_2']
    condition = session['condition']

    curated_hours_info = session['curated_hours_info'].copy()

    print(f'Processing session: {session_id}; mouse_1: {mouse_1}; condition: {condition}')

    for i_hour, curated_hour in curated_hours_info.iterrows():

        df_windows = session['curated_hours_l_windows'][i_hour]

        # add metadata
        df_windows['mouse_id'] = mouse_1
        df_windows['condition'] = condition
        df_windows['session_id'] = session_id
        df_windows['file_idx'] = curated_hour['file_idx']
        if condition == 'pre':
            df_windows['full_condition'] = 'pre'
        else:
            post_period = curated_hour['post_period']
            print(f'file_idx: {curated_hour["file_idx"]}; post_period: {post_period}')
            df_windows['full_condition'] = f'post_{post_period}'

        # save
        all_curated_hours_windows.append(df_windows)

df_all_curated_hours_windows = pd.concat(all_curated_hours_windows, ignore_index=True)

In [ ]:
# collect all segments
all_curated_hours_segments = []

for session in all_sessions:

    session_id = session['session_id']
    mouse_1 = session['mouse_1']
    mouse_2 = session['mouse_2']
    condition = session['condition']

    curated_hours_info = session['curated_hours_info'].copy()

    print(f'Processing session: {session_id}; mouse_1: {mouse_1}; condition: {condition}')

    for i_hour, curated_hour in curated_hours_info.iterrows():
        # print(f'File index: {curated_hour["file_idx"]}; hour offset: {curated_hour["hour_offset"]:.2f}')
        df_segments = session['curated_hours_l_segments'][i_hour]

        # add metadata
        df_segments['mouse_id'] = mouse_1
        df_segments['condition'] = condition
        df_segments['session_id'] = session_id
        df_segments['file_idx'] = curated_hour['file_idx']
        if condition == 'pre':
            df_segments['full_condition'] = 'pre'
        else:
            post_period = curated_hour['post_period']
            print(f'file_idx: {curated_hour["file_idx"]}; post_period: {post_period}')
            df_segments['full_condition'] = f'post_{post_period}'

        # save
        all_curated_hours_segments.append(df_segments)

df_all_curated_hours_segments = pd.concat(all_curated_hours_segments, ignore_index=True)

In [ ]:
# collect all song bouts
all_curated_hours_bouts_song = []

for session in all_sessions:

    session_id = session['session_id']
    mouse_1 = session['mouse_1']
    mouse_2 = session['mouse_2']
    condition = session['condition']

    curated_hours_info = session['curated_hours_info'].copy()

    print(f'Processing session: {session_id}; mouse_1: {mouse_1}; condition: {condition}')

    for i_hour, curated_hour in curated_hours_info.iterrows():
        # print(f'File index: {curated_hour["file_idx"]}; hour offset: {curated_hour["hour_offset"]:.2f}')
        df_bouts = session['curated_hours_l_vocal_bouts_song'][i_hour]

        # add metadata
        df_bouts['mouse_id'] = mouse_1
        df_bouts['condition'] = condition
        df_bouts['session_id'] = session_id
        df_bouts['file_idx'] = curated_hour['file_idx']
        if condition == 'pre':
            df_bouts['full_condition'] = 'pre'
        else:
            post_period = curated_hour['post_period']
            print(f'file_idx: {curated_hour["file_idx"]}; post_period: {post_period}')
            df_bouts['full_condition'] = f'post_{post_period}'

        # save
        all_curated_hours_bouts_song.append(df_bouts)

df_all_curated_hours_bouts_song = pd.concat(all_curated_hours_bouts_song, ignore_index=True)

In [ ]:
# collect all usv bouts
all_curated_hours_bouts_usv = []

for session in all_sessions:

    session_id = session['session_id']
    mouse_1 = session['mouse_1']
    mouse_2 = session['mouse_2']
    condition = session['condition']

    curated_hours_info = session['curated_hours_info'].copy()

    print(f'Processing session: {session_id}; mouse_1: {mouse_1}; condition: {condition}')

    for i_hour, curated_hour in curated_hours_info.iterrows():
        # print(f'File index: {curated_hour["file_idx"]}; hour offset: {curated_hour["hour_offset"]:.2f}')
        df_bouts = session['curated_hours_l_vocal_bouts_usv'][i_hour]

        # add metadata
        df_bouts['mouse_id'] = mouse_1
        df_bouts['condition'] = condition
        df_bouts['session_id'] = session_id
        df_bouts['file_idx'] = curated_hour['file_idx']
        if condition == 'pre':
            df_bouts['full_condition'] = 'pre'
        else:
            post_period = curated_hour['post_period']
            print(f'file_idx: {curated_hour["file_idx"]}; post_period: {post_period}')
            df_bouts['full_condition'] = f'post_{post_period}'

        # save
        all_curated_hours_bouts_usv.append(df_bouts)

df_all_curated_hours_bouts_usv = pd.concat(all_curated_hours_bouts_usv, ignore_index=True)

In [ ]:
# notes
df_all_curated_hours_notes_pre = df_all_curated_hours_notes[df_all_curated_hours_notes['full_condition']=='pre']
df_all_curated_hours_notes_post_early = df_all_curated_hours_notes[df_all_curated_hours_notes['full_condition']=='post_early']
df_all_curated_hours_notes_post_late = df_all_curated_hours_notes[df_all_curated_hours_notes['full_condition']=='post_late']

# windows
df_all_curated_hours_windows_pre = df_all_curated_hours_windows[df_all_curated_hours_windows['full_condition']=='pre']
df_all_curated_hours_windows_post_early = df_all_curated_hours_windows[df_all_curated_hours_windows['full_condition']=='post_early']
df_all_curated_hours_windows_post_late = df_all_curated_hours_windows[df_all_curated_hours_windows['full_condition']=='post_late']

# long segments
df_all_curated_hours_segments_pre = df_all_curated_hours_segments[df_all_curated_hours_segments['full_condition']=='pre']
df_all_curated_hours_segments_post_early = df_all_curated_hours_segments[df_all_curated_hours_segments['full_condition']=='post_early']
df_all_curated_hours_segments_post_late = df_all_curated_hours_segments[df_all_curated_hours_segments['full_condition']=='post_late']

In [ ]:
# local windows - pre

# plot local patterning features - local window nrmse vs gap

# Compute KDE for all notes
data_all = df_all_curated_hours_windows_pre[['window_max_log_inter_event_interval', 'window_rate_index_normalized_rmse']].values.T
x_all, y_all, Z_all = compute_2d_kde(
    data=data_all,
    x_min=-2,
    x_max=1,
    y_min=0,
    y_max=1,
    bw_method='scott',
    gridsize=100,
)

# fig, ax = plt.subplots(figsize=(4,4))
fig, ax = plt.subplots(figsize=(2.25, 2))

# all notes
# kde heatmap
heatmap_all = ax.imshow(
    Z_all.T,
    origin='lower',
    aspect='auto',
    extent=[x_all.min(), x_all.max(), y_all.min(), y_all.max()],
    cmap='Blues',
    vmin=0, vmax=3,
)
cbar_main = plt.colorbar(heatmap_all, ax=ax, label='Density')

# kde contour plot
sns.kdeplot(data=df_all_curated_hours_windows_pre,
            x='window_max_log_inter_event_interval', 
            y='window_rate_index_normalized_rmse',
            thresh=0.1, levels=5, bw_adjust=1,
            clip = [[-2, 1], [0, 1]],
            gridsize=100, fill=False, color='black',
            linewidths=1,
            ax=ax)

# decision boundary for local stereotypy window
ax.add_patch(Rectangle((-2, 0), 
                       window_max_log_isi_threshold-(-2), window_rmse_threshold, linewidth=2, edgecolor='gray', linestyle='--', facecolor='none', zorder=3))

# Set limits and labels for main axes
ax.set_xlim([-2, 1])
ax.set_ylim([0, 1])
# ax.set_title('Social - fixed-size window NRMSE vs max ISI')
ax.set_xlabel('Local Max ISI (log second)')
ax.set_ylabel('Local Normalized RMSE')
plt.tight_layout()
# plt.show()

In [ ]:
# local windows - post early

# plot local patterning features - local window nrmse vs gap

# Compute KDE for all notes
data_all = df_all_curated_hours_windows_post_early[['window_max_log_inter_event_interval', 'window_rate_index_normalized_rmse']].values.T
x_all, y_all, Z_all = compute_2d_kde(
    data=data_all,
    x_min=-2,
    x_max=1,
    y_min=0,
    y_max=1,
    bw_method='scott',
    gridsize=100,
)

# fig, ax = plt.subplots(figsize=(4,4))
fig, ax = plt.subplots(figsize=(2.25, 2))

# all notes
# kde heatmap
heatmap_all = ax.imshow(
    Z_all.T,
    origin='lower',
    aspect='auto',
    extent=[x_all.min(), x_all.max(), y_all.min(), y_all.max()],
    cmap='Oranges',
    vmin=0, vmax=3,
)
cbar_main = plt.colorbar(heatmap_all, ax=ax, label='Density')

# kde contour plot
sns.kdeplot(data=df_all_curated_hours_windows_post_early,
            x='window_max_log_inter_event_interval', 
            y='window_rate_index_normalized_rmse',
            thresh=0.1, levels=5, bw_adjust=1,
            clip = [[-2, 1], [0, 1]],
            gridsize=100, fill=False, color='black',
            linewidths=1,
            ax=ax)

# decision boundary for local stereotypy window
ax.add_patch(Rectangle((-2, 0), 
                       window_max_log_isi_threshold-(-2), window_rmse_threshold, linewidth=2, edgecolor='gray', linestyle='--', facecolor='none', zorder=3))

# Set limits and labels for main axes
ax.set_xlim([-2, 1])
ax.set_ylim([0, 1])
# ax.set_title('Social - fixed-size window NRMSE vs max ISI')
ax.set_xlabel('Local Max ISI (log second)')
ax.set_ylabel('Local Normalized RMSE')
plt.tight_layout()
# plt.show()

In [ ]:
# local windows - post late

# plot local patterning features - local window nrmse vs gap

# Compute KDE for all notes
data_all = df_all_curated_hours_windows_post_late[['window_max_log_inter_event_interval', 'window_rate_index_normalized_rmse']].values.T
x_all, y_all, Z_all = compute_2d_kde(
    data=data_all,
    x_min=-2,
    x_max=1,
    y_min=0,
    y_max=1,
    bw_method='scott',
    gridsize=100,
)

# fig, ax = plt.subplots(figsize=(4,4))
fig, ax = plt.subplots(figsize=(2.25, 2))

# all notes
# kde heatmap
heatmap_all = ax.imshow(
    Z_all.T,
    origin='lower',
    aspect='auto',
    extent=[x_all.min(), x_all.max(), y_all.min(), y_all.max()],
    cmap='Greens',
    vmin=0, vmax=3,
)
cbar_main = plt.colorbar(heatmap_all, ax=ax, label='Density')

# kde contour plot
sns.kdeplot(data=df_all_curated_hours_windows_post_late,
            x='window_max_log_inter_event_interval', 
            y='window_rate_index_normalized_rmse',
            thresh=0.1, levels=5, bw_adjust=1,
            clip = [[-2, 1], [0, 1]],
            gridsize=100, fill=False, color='black',
            linewidths=1,
            ax=ax)

# decision boundary for local stereotypy window
ax.add_patch(Rectangle((-2, 0), 
                       window_max_log_isi_threshold-(-2), window_rmse_threshold, linewidth=2, edgecolor='gray', linestyle='--', facecolor='none', zorder=3))

# Set limits and labels for main axes
ax.set_xlim([-2, 1])
ax.set_ylim([0, 1])
# ax.set_title('Social - fixed-size window NRMSE vs max ISI')
ax.set_xlabel('Local Max ISI (log second)')
ax.set_ylabel('Local Normalized RMSE')
plt.tight_layout()
# plt.show()

In [ ]:
# segments - pre

# Define the feature columns expected by the models
features = ['ransac_num_events', 'ransac_max_rate', 'ransac_min_rate']

# Ensure the dataframe contains the required features
missing_features = [feat for feat in features if feat not in df_all_curated_hours_segments.columns]
if missing_features:
    raise ValueError(f"Missing required features: {missing_features}")

# Select and copy the relevant data
data = df_all_curated_hours_segments_pre[features].copy()

# Apply the scaler and PCA transformations
X_scaled = scaler.transform(data)
X_pca = pca.transform(X_scaled)

# Predict cluster labels using the GMM model
labels = gmm.predict(X_pca)

# Map numeric labels to semantic labels
mapped_labels = np.array([cluster_mapping[label] for label in labels])

# Create a palette mapping for the clusters present in the data
palette_label = {label: color_palette_patterning[label] for label in np.unique(mapped_labels)}

# make 2d-kde
data_all = X_pca.T
x_all, y_all, Z_all = compute_2d_kde(
    data=data_all,
    x_min=-2.75,
    x_max=2.75,
    y_min=-3.5,
    y_max=4.25,
    bw_method='scott',
    gridsize=100,
)

# plot the clusters in pc space
fig, ax = plt.subplots(figsize=(2.25, 2))
# kde heatmap
heatmap_all = ax.imshow(
    Z_all.T,
    origin='lower',
    aspect='auto',
    extent=[x_all.min(), x_all.max(), y_all.min(), y_all.max()],
    cmap='Blues',
    vmin=0, vmax=0.16,
)
cbar_main = plt.colorbar(heatmap_all, ax=ax, label='Density')


sns.kdeplot(x=X_pca[:, 0], y=X_pca[:, 1], 
            color='k', label='pre',
            thresh=0.1, levels=5, bw_adjust=1, linewidths=1, legend=None, ax=ax)
# Overlay ellipses for each Gaussian component for 2σ
for k, (mean, covar) in enumerate(zip(gmm.means_, gmm.covariances_)):
    cluster_label = cluster_mapping[k]  # get the semantic label for component k
    cluster_color = color_palette_patterning[cluster_label]
    draw_ellipse(mean, covar, ax, n_std=1, edgecolor=cluster_color, facecolor='none', lw=1)
    draw_ellipse(mean, covar, ax, n_std=2, edgecolor=cluster_color, facecolor='none', lw=1)
    # draw_ellipse(mean, covar, ax, n_std=3, edgecolor=cluster_color, facecolor='none', lw=0.5)
# # Plot centroids
# for k, mean in enumerate(gmm.means_):
#     ax.scatter(mean[0], mean[1], color='black', marker='X', s=50, edgecolor='none')

# ax.set_title('Backbone of long segments')
# ax.legend(loc='upper left', fontsize='x-small')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_xlim([-2.75, 2.75])
ax.set_ylim([-3.5, 4.25])
plt.tight_layout()
# plt.show()

In [ ]:
# segments - post-early

# Define the feature columns expected by the models
features = ['ransac_num_events', 'ransac_max_rate', 'ransac_min_rate']

# Ensure the dataframe contains the required features
missing_features = [feat for feat in features if feat not in df_all_curated_hours_segments.columns]
if missing_features:
    raise ValueError(f"Missing required features: {missing_features}")

# Select and copy the relevant data
data = df_all_curated_hours_segments_post_early[features].copy()

# Apply the scaler and PCA transformations
X_scaled = scaler.transform(data)
X_pca = pca.transform(X_scaled)

# Predict cluster labels using the GMM model
labels = gmm.predict(X_pca)

# Map numeric labels to semantic labels
mapped_labels = np.array([cluster_mapping[label] for label in labels])

# Create a palette mapping for the clusters present in the data
palette_label = {label: color_palette_patterning[label] for label in np.unique(mapped_labels)}

# make 2d-kde
data_all = X_pca.T
x_all, y_all, Z_all = compute_2d_kde(
    data=data_all,
    x_min=-2.75,
    x_max=2.75,
    y_min=-3.5,
    y_max=4.25,
    bw_method='scott',
    gridsize=100,
)

# plot the clusters in pc space
fig, ax = plt.subplots(figsize=(2.25, 2))
# kde heatmap
heatmap_all = ax.imshow(
    Z_all.T,
    origin='lower',
    aspect='auto',
    extent=[x_all.min(), x_all.max(), y_all.min(), y_all.max()],
    cmap='Oranges',
    vmin=0, vmax=0.16,
)
cbar_main = plt.colorbar(heatmap_all, ax=ax, label='Density')


sns.kdeplot(x=X_pca[:, 0], y=X_pca[:, 1], 
            color='k', label='pre',
            thresh=0.1, levels=5, bw_adjust=1, linewidths=1, legend=None, ax=ax)
# Overlay ellipses for each Gaussian component for 2σ
for k, (mean, covar) in enumerate(zip(gmm.means_, gmm.covariances_)):
    cluster_label = cluster_mapping[k]  # get the semantic label for component k
    cluster_color = color_palette_patterning[cluster_label]
    draw_ellipse(mean, covar, ax, n_std=1, edgecolor=cluster_color, facecolor='none', lw=1)
    draw_ellipse(mean, covar, ax, n_std=2, edgecolor=cluster_color, facecolor='none', lw=1)
    # draw_ellipse(mean, covar, ax, n_std=3, edgecolor=cluster_color, facecolor='none', lw=0.5)
# # Plot centroids
# for k, mean in enumerate(gmm.means_):
#     ax.scatter(mean[0], mean[1], color='black', marker='X', s=50, edgecolor='none')

# ax.set_title('Backbone of long segments')
# ax.legend(loc='upper left', fontsize='x-small')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_xlim([-2.75, 2.75])
ax.set_ylim([-3.5, 4.25])
plt.tight_layout()
# plt.show()

In [ ]:
# segments - post-late

# Define the feature columns expected by the models
features = ['ransac_num_events', 'ransac_max_rate', 'ransac_min_rate']

# Ensure the dataframe contains the required features
missing_features = [feat for feat in features if feat not in df_all_curated_hours_segments.columns]
if missing_features:
    raise ValueError(f"Missing required features: {missing_features}")

# Select and copy the relevant data
data = df_all_curated_hours_segments_post_late[features].copy()

# Apply the scaler and PCA transformations
X_scaled = scaler.transform(data)
X_pca = pca.transform(X_scaled)

# Predict cluster labels using the GMM model
labels = gmm.predict(X_pca)

# Map numeric labels to semantic labels
mapped_labels = np.array([cluster_mapping[label] for label in labels])

# Create a palette mapping for the clusters present in the data
palette_label = {label: color_palette_patterning[label] for label in np.unique(mapped_labels)}

# make 2d-kde
data_all = X_pca.T
x_all, y_all, Z_all = compute_2d_kde(
    data=data_all,
    x_min=-2.75,
    x_max=2.75,
    y_min=-3.5,
    y_max=4.25,
    bw_method='scott',
    gridsize=100,
)

# plot the clusters in pc space
fig, ax = plt.subplots(figsize=(2.25, 2))
# kde heatmap
heatmap_all = ax.imshow(
    Z_all.T,
    origin='lower',
    aspect='auto',
    extent=[x_all.min(), x_all.max(), y_all.min(), y_all.max()],
    cmap='Greens',
    vmin=0, vmax=0.16,
)
cbar_main = plt.colorbar(heatmap_all, ax=ax, label='Density')


sns.kdeplot(x=X_pca[:, 0], y=X_pca[:, 1], 
            color='k', label='pre',
            thresh=0.1, levels=5, bw_adjust=1, linewidths=1, legend=None, ax=ax)
# Overlay ellipses for each Gaussian component for 2σ
for k, (mean, covar) in enumerate(zip(gmm.means_, gmm.covariances_)):
    cluster_label = cluster_mapping[k]  # get the semantic label for component k
    cluster_color = color_palette_patterning[cluster_label]
    draw_ellipse(mean, covar, ax, n_std=1, edgecolor=cluster_color, facecolor='none', lw=1)
    draw_ellipse(mean, covar, ax, n_std=2, edgecolor=cluster_color, facecolor='none', lw=1)
    # draw_ellipse(mean, covar, ax, n_std=3, edgecolor=cluster_color, facecolor='none', lw=0.5)
# # Plot centroids
# for k, mean in enumerate(gmm.means_):
#     ax.scatter(mean[0], mean[1], color='black', marker='X', s=50, edgecolor='none')

# ax.set_title('Backbone of long segments')
# ax.legend(loc='upper left', fontsize='x-small')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_xlim([-2.75, 2.75])
ax.set_ylim([-3.5, 4.25])
plt.tight_layout()
# plt.show()

In [ ]:
# quantify number of song notes, semi-song notes, and USVs, and their proportions in each epoch
vocal_modes_condition_summary = []
for focal_mouse_id in df_all_curated_hours_notes['mouse_id'].unique():
    df_focal_mouse_notes = df_all_curated_hours_notes[df_all_curated_hours_notes['mouse_id']==focal_mouse_id]
    for condition in ['pre', 'post_early', 'post_late']:
        df_focal_mouse_condition_notes = df_focal_mouse_notes[df_focal_mouse_notes['full_condition']==condition]
        num_notes = len(df_focal_mouse_condition_notes)
        num_usvs = len(df_focal_mouse_condition_notes[df_focal_mouse_condition_notes['type_final']=='USV'])
        num_song_notes = len(df_focal_mouse_condition_notes[df_focal_mouse_condition_notes['type_final']=='song'])
        num_semisong_notes = len(df_focal_mouse_condition_notes[df_focal_mouse_condition_notes['type_final']=='proto-song'])

        print(f'focal_mouse_id: {focal_mouse_id}; condition: {condition}; num_notes: {num_notes}; num_song_notes: {num_song_notes}; num_semisong_notes: {num_semisong_notes}; num_usvs: {num_usvs}')
        vocal_modes_condition_summary.append({'mouse_id': focal_mouse_id, 
                                              'condition': condition, 
                                              'num_song_notes': num_song_notes, 
                                              'num_semisong_notes': num_semisong_notes,
                                              'num_usvs': num_usvs, 
                                              'num_notes': num_notes})

df_vocal_modes_condition_summary = pd.DataFrame(vocal_modes_condition_summary)

# compute percentage
df_vocal_modes_condition_summary['perc_song_notes'] = df_vocal_modes_condition_summary['num_song_notes'] / df_vocal_modes_condition_summary['num_notes'] * 100
df_vocal_modes_condition_summary['perc_semisong_notes'] = df_vocal_modes_condition_summary['num_semisong_notes'] / df_vocal_modes_condition_summary['num_notes'] * 100
df_vocal_modes_condition_summary['perc_usvs'] = df_vocal_modes_condition_summary['num_usvs'] / df_vocal_modes_condition_summary['num_notes'] * 100
print('Summary:')
print(f'Sum of USV notes: {df_vocal_modes_condition_summary["num_usvs"].sum()}')
print(f'Sum of Semisong notes: {df_vocal_modes_condition_summary["num_semisong_notes"].sum()}')
print(f'Sum of Song notes: {df_vocal_modes_condition_summary["num_song_notes"].sum()}')
print(f'Sum of all notes: {df_vocal_modes_condition_summary["num_notes"].sum()}')
print(f'Sum of pre notes: {df_vocal_modes_condition_summary[df_vocal_modes_condition_summary["condition"]=="pre"]["num_notes"].sum()}')
print(f'Sum of post-early notes: {df_vocal_modes_condition_summary[df_vocal_modes_condition_summary["condition"]=="post_early"]["num_notes"].sum()}')
print(f'Sum of post-late notes: {df_vocal_modes_condition_summary[df_vocal_modes_condition_summary["condition"]=="post_late"]["num_notes"].sum()}')

fig, ax = plt.subplots(figsize=(1.5, 1.5))

# plot mean and sem percentage of notes as a function of epoch

# usv
pre_perc_usv_mean = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='pre']['perc_usvs'].mean()
pre_perc_usv_sem = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='pre']['perc_usvs'].sem()
post_early_perc_usv_mean = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_early']['perc_usvs'].mean()
post_early_perc_usv_sem = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_early']['perc_usvs'].sem()
post_late_perc_usv_mean = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_late']['perc_usvs'].mean()
post_late_perc_usv_sem = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_late']['perc_usvs'].sem()
print(f'Pre USV: {pre_perc_usv_mean:.2f} ± {pre_perc_usv_sem:.2f}')
print(f'Post-early USV: {post_early_perc_usv_mean:.2f} ± {post_early_perc_usv_sem:.2f}')
print(f'Post-late USV: {post_late_perc_usv_mean:.2f} ± {post_late_perc_usv_sem:.2f}')

# semisong notes
pre_perc_semisong_mean = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='pre']['perc_semisong_notes'].mean()
pre_perc_semisong_sem = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='pre']['perc_semisong_notes'].sem()
post_early_perc_semisong_mean = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_early']['perc_semisong_notes'].mean()
post_early_perc_semisong_sem = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_early']['perc_semisong_notes'].sem()
post_late_perc_semisong_mean = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_late']['perc_semisong_notes'].mean()
post_late_perc_semisong_sem = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_late']['perc_semisong_notes'].sem()
print(f'Pre semisong: {pre_perc_semisong_mean:.2f} ± {pre_perc_semisong_sem:.2f}')
print(f'Post-early semisong: {post_early_perc_semisong_mean:.2f} ± {post_early_perc_semisong_sem:.2f}')
print(f'Post-late semisong: {post_late_perc_semisong_mean:.2f} ± {post_late_perc_semisong_sem:.2f}')

# song notes
pre_perc_song_mean = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='pre']['perc_song_notes'].mean()
pre_perc_song_sem = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='pre']['perc_song_notes'].sem()
post_early_perc_song_mean = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_early']['perc_song_notes'].mean()
post_early_perc_song_sem = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_early']['perc_song_notes'].sem()
post_late_perc_song_mean = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_late']['perc_song_notes'].mean()
post_late_perc_song_sem = df_vocal_modes_condition_summary[df_vocal_modes_condition_summary['condition']=='post_late']['perc_song_notes'].sem()
print(f'Pre song: {pre_perc_song_mean:.2f} ± {pre_perc_song_sem:.2f}')
print(f'Post-early song: {post_early_perc_song_mean:.2f} ± {post_early_perc_song_sem:.2f}')
print(f'Post-late song: {post_late_perc_song_mean:.2f} ± {post_late_perc_song_sem:.2f}')

# plot
# usv
ax.errorbar([0, 1, 2], [pre_perc_usv_mean, post_early_perc_usv_mean, post_late_perc_usv_mean], 
            yerr=[pre_perc_usv_sem, post_early_perc_usv_sem, post_late_perc_usv_sem], 
            fmt='-o', markersize=2.5, color='tab:purple')
# semisong notes
ax.errorbar([0, 1, 2], [pre_perc_semisong_mean, post_early_perc_semisong_mean, post_late_perc_semisong_mean], 
            yerr=[pre_perc_semisong_sem, post_early_perc_semisong_sem, post_late_perc_semisong_sem], 
            fmt='-o', markersize=2.5, color='tab:blue')
# song notes
ax.errorbar([0, 1, 2], [pre_perc_song_mean, post_early_perc_song_mean, post_late_perc_song_mean],
            yerr=[pre_perc_song_sem, post_early_perc_song_sem, post_late_perc_song_sem],
            fmt='-o', markersize=2.5, color='tab:cyan')

ax.set_ylim([0, 100])
ax.set_yticks([0, 50, 100])
ax.set_ylabel('Vocalizations (%)')
ax.set_xlim([-0.25, 2.25])
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Pre', 'Post-early', 'Post-late'])

plt.tight_layout()

plt.show()



In [ ]:
# for USVs and song notes, plot loudness of each mode, relative

# USV
df_all_curated_hours_usvs = df_all_curated_hours_notes[df_all_curated_hours_notes['type_final']=='USV'].copy()
print('USVs:')
print(df_all_curated_hours_usvs['full_condition'].value_counts())

# compute relative loudness
pre_usv_amp_mean = df_all_curated_hours_usvs[df_all_curated_hours_usvs['full_condition']=='pre']['max_amp_high'].mean()
print(f'Pre USV mean amp (high): {pre_usv_amp_mean:.2f}')
df_all_curated_hours_usvs['rel_max_amp_high'] = df_all_curated_hours_usvs['max_amp_high'] - pre_usv_amp_mean
pre_usv_rel_amp_mean = df_all_curated_hours_usvs[df_all_curated_hours_usvs['full_condition']=='pre']['rel_max_amp_high'].mean()
pre_usv_rel_amp_sem = df_all_curated_hours_usvs[df_all_curated_hours_usvs['full_condition']=='pre']['rel_max_amp_high'].sem()
post_early_usv_rel_amp_mean = df_all_curated_hours_usvs[df_all_curated_hours_usvs['full_condition']=='post_early']['rel_max_amp_high'].mean()
post_early_usv_rel_amp_sem = df_all_curated_hours_usvs[df_all_curated_hours_usvs['full_condition']=='post_early']['rel_max_amp_high'].sem()
post_late_usv_rel_amp_mean = df_all_curated_hours_usvs[df_all_curated_hours_usvs['full_condition']=='post_late']['rel_max_amp_high'].mean()
post_late_usv_rel_amp_sem = df_all_curated_hours_usvs[df_all_curated_hours_usvs['full_condition']=='post_late']['rel_max_amp_high'].sem()
print(f'USV - Pre: {pre_usv_rel_amp_mean:.2f} ± {pre_usv_rel_amp_sem:.2f}')
print(f'USV - Post Early: {post_early_usv_rel_amp_mean:.2f} ± {post_early_usv_rel_amp_sem:.2f}')
print(f'USV - Post Late: {post_late_usv_rel_amp_mean:.2f} ± {post_late_usv_rel_amp_sem:.2f}')

# song notes
df_all_curated_hours_song_notes = df_all_curated_hours_notes[df_all_curated_hours_notes['type_final']=='song'].copy()
print('Song notes:')
print(df_all_curated_hours_song_notes['full_condition'].value_counts())

pre_song_amp_mean = df_all_curated_hours_song_notes[df_all_curated_hours_song_notes['full_condition']=='pre']['max_amp_song'].mean()
print(f'Pre song mean amp (low): {pre_song_amp_mean:.2f}')
df_all_curated_hours_song_notes['rel_max_amp_song'] = df_all_curated_hours_song_notes['max_amp_song'] - pre_song_amp_mean
pre_song_rel_amp_mean = df_all_curated_hours_song_notes[df_all_curated_hours_song_notes['full_condition']=='pre']['rel_max_amp_song'].mean()
pre_song_rel_amp_sem = df_all_curated_hours_song_notes[df_all_curated_hours_song_notes['full_condition']=='pre']['rel_max_amp_song'].sem()
post_early_song_rel_amp_mean = df_all_curated_hours_song_notes[df_all_curated_hours_song_notes['full_condition']=='post_early']['rel_max_amp_song'].mean()
post_early_song_rel_amp_sem = df_all_curated_hours_song_notes[df_all_curated_hours_song_notes['full_condition']=='post_early']['rel_max_amp_song'].sem()
post_late_song_rel_amp_mean = df_all_curated_hours_song_notes[df_all_curated_hours_song_notes['full_condition']=='post_late']['rel_max_amp_song'].mean()
post_late_song_rel_amp_sem = df_all_curated_hours_song_notes[df_all_curated_hours_song_notes['full_condition']=='post_late']['rel_max_amp_song'].sem()
print(f'Song - Pre: {pre_song_rel_amp_mean:.2f} ± {pre_song_rel_amp_sem:.2f}')
print(f'Song - Post Early: {post_early_song_rel_amp_mean:.2f} ± {post_early_song_rel_amp_sem:.2f}')
print(f'Song - Post Late: {post_late_song_rel_amp_mean:.2f} ± {post_late_song_rel_amp_sem:.2f}')

# plot
fig, axes = plt.subplots(figsize=(2.5, 1.5), nrows=1, ncols=2, sharex=True, sharey=True)

# usv
sns.violinplot(
    x='full_condition',
    y='rel_max_amp_high',
    data=df_all_curated_hours_usvs,
    hue='full_condition',
    palette={'pre':'tab:blue', 'post_early':'tab:orange', 'post_late':'tab:green'},
    ax=axes[0]
)

# song
sns.violinplot(
    x='full_condition',
    y='rel_max_amp_song',
    data=df_all_curated_hours_song_notes,
    hue='full_condition',
    palette={'pre':'tab:blue', 'post_early':'tab:orange', 'post_late':'tab:green'},
    ax=axes[1]
)

axes[0].set_ylabel('Relative amplitude (dB)')
axes[1].set_ylim([-35, 35])
axes[0].set_xlabel('')
axes[1].set_xlabel('')

fig.align_labels(axes)
plt.tight_layout()

plt.show()

## Use song model to analyze perturbed songs

In [ ]:
# collect all songs
all_curated_hours_songs = []

for session in all_sessions:

    session_id = session['session_id']
    mouse_1 = session['mouse_1']
    mouse_2 = session['mouse_2']
    condition = session['condition']

    curated_hours_info = session['curated_hours_info'].copy()

    print(f'Processing session: {session_id}; mouse_1: {mouse_1}; condition: {condition}')

    for i_hour, curated_hour in curated_hours_info.iterrows():

        df_songs = session['curated_hours_l_songs'][i_hour]

        # add metadata
        df_songs['mouse_id'] = mouse_1
        df_songs['condition'] = condition
        df_songs['session_id'] = session_id
        df_songs['file_idx'] = curated_hour['file_idx']
        if condition == 'pre':
            df_songs['full_condition'] = 'pre'
        else:
            post_period = curated_hour['post_period']
            print(f'file_idx: {curated_hour["file_idx"]}; post_period: {post_period}')
            df_songs['full_condition'] = f'post_{post_period}'

        # save
        all_curated_hours_songs.append(df_songs)

df_all_curated_hours_songs = pd.concat(all_curated_hours_songs, ignore_index=True)

In [ ]:
# Compare song duration between male and female
fig, ax = plt.subplots(figsize=(1.75, 1.75))

sns.violinplot(df_all_curated_hours_songs,
              x='full_condition',
              y='duration',
              order=['pre', 'post_early', 'post_late'],
              hue='full_condition', palette={'pre':'tab:blue', 'post_early':'tab:orange', 'post_late':'tab:green'}, hue_order=['pre', 'post_early', 'post_late'],
              ax=ax)

ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Pre', 'Post-early', 'Post-late'])
ax.set_ylim([0,16])
ax.set_xlabel('')
ax.set_ylabel('Song duration (s)')

plt.tight_layout()

# plt.show()

In [ ]:
print(df_all_curated_hours_songs.groupby('full_condition')[['duration', 'rate_idx_slope', 'rate_idx_rmax', 'rate_idx_rmin']].agg(['count', 'mean', 'sem']).T)

In [ ]:
fig, axes = plt.subplots(figsize=(4, 1.75), ncols=3)

# slope
sns.violinplot(df_all_curated_hours_songs,
              x='full_condition',
              y='rate_idx_slope',
              order=['pre', 'post_early', 'post_late'],
              hue='full_condition', palette={'pre':'tab:blue', 'post_early':'tab:orange', 'post_late':'tab:green'}, hue_order=['pre', 'post_early', 'post_late'],
              legend=False,
              ax=axes[0])
axes[0].set_ylabel('Slope (Hz)')
axes[0].set_ylim([-0.5, 0])
axes[0].set_xlabel('')

# rmax
sns.violinplot(df_all_curated_hours_songs,
              x='full_condition',
              y='rate_idx_rmax',
              order=['pre', 'post_early', 'post_late'],
              hue='full_condition', palette={'pre':'tab:blue', 'post_early':'tab:orange', 'post_late':'tab:green'}, hue_order=['pre', 'post_early', 'post_late'],
              legend=False,
              ax=axes[1])
axes[1].set_ylabel('r_max (Hz)')
axes[1].set_ylim([0, 30])
axes[1].set_xlabel('')

# rmax
sns.violinplot(df_all_curated_hours_songs,
              x='full_condition',
              y='rate_idx_rmin',
              order=['pre', 'post_early', 'post_late'],
              hue='full_condition', palette={'pre':'tab:blue', 'post_early':'tab:orange', 'post_late':'tab:green'}, hue_order=['pre', 'post_early', 'post_late'],
              legend=False,
              ax=axes[2])
axes[2].set_ylabel('r_min (Hz)')
axes[2].set_ylim([0, 30])
axes[2].set_xlabel('')

plt.tight_layout()

# plt.show()

In [ ]:
# Compute the model values as before
mean_rmax = df_all_curated_hours_songs['rate_idx_rmax'].mean()
mean_slope = df_all_curated_hours_songs['rate_idx_slope'].mean()
print(f'Mean r_max: {mean_rmax:.2f}')
print(f'Mean slope: {mean_slope:.4f}')

xs = np.arange(4, 18, 0.1)
ys = 1 / mean_slope * np.log(xs / mean_rmax) + 1 / xs * song_final_note_occupancy

# Create a JointGrid with custom limits
g = sns.JointGrid(data=df_all_curated_hours_songs, 
                  x='rate_idx_rmin', 
                  y='duration', 
                  xlim=(0, 22.5), 
                  ylim=(0, 16), 
                  marginal_ticks=True,
                  height=2, ratio=4)

# Plot the joint scatter plot with hue by full_condition
scatter = sns.scatterplot(data=df_all_curated_hours_songs,
                          x='rate_idx_rmin',
                          y='duration',
                          hue='full_condition',
                          palette={'pre':'tab:blue', 'post_early':'tab:orange', 'post_late':'tab:green'},
                          hue_order=['pre', 'post_early', 'post_late'],
                          s=2,
                          linewidth=0,
                          ax=g.ax_joint)

# Plot the model curve on the joint axes
g.ax_joint.plot(xs, ys, color='k', linewidth=1, linestyle='--', label='Model')

# Add the marginal KDE plots
sns.kdeplot(data=df_all_curated_hours_songs, 
            x='rate_idx_rmin', 
            hue='full_condition',
            palette={'pre':'tab:blue', 'post_early':'tab:orange', 'post_late':'tab:green'},
            hue_order=['pre', 'post_early', 'post_late'],
            fill=True, common_norm=False, legend=None,
            ax=g.ax_marg_x)
sns.kdeplot(data=df_all_curated_hours_songs, 
            y='duration', 
            hue='full_condition',
            palette={'pre':'tab:blue', 'post_early':'tab:orange', 'post_late':'tab:green'},
            hue_order=['pre', 'post_early', 'post_late'],
            fill=True, common_norm=False, legend=None,
            ax=g.ax_marg_y)

# Set axis labels
g.set_axis_labels('r_min (Hz)', 'Song duration (s)')

# Add a legend to the joint plot (adjusting font sizes as needed)
g.ax_joint.legend(title='', fontsize='x-small', title_fontsize='x-small', loc='upper right')

plt.tight_layout()
# plt.show()

In [ ]:
# Compare longest note in each song
fig, ax = plt.subplots(figsize=(1.75, 1.75))

sns.violinplot(df_all_curated_hours_songs,
              x='full_condition',
              y='longest_note_duration',
              order=['pre', 'post_early', 'post_late'],
              hue='full_condition', palette={'pre':'tab:blue', 'post_early':'tab:orange', 'post_late':'tab:green'}, hue_order=['pre', 'post_early', 'post_late'],
              ax=ax)

ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Pre', 'Post-early', 'Post-late'])
ax.set_ylim([0,0.15])
ax.set_xlabel('')
ax.set_ylabel('Duration of longest note (s)')

plt.tight_layout()
# plt.show()